## ganda-grammar, carried by this notebook

This copy of `build_wordlist.ipynb` carries ganda-grammar (commit c92115b). The cells below write its package (`src/`, as
`ganda_grammar`) and `scripts/` to `/tmp/ganda_grammar_bundle` and put that first on `sys.path`, so the notebook needs no copy of the
repository as an input. Made by `scripts/bundle_notebook.py`; rebuild it after changing the code.


In [ ]:
import sys
from pathlib import Path

BUNDLE = Path("/tmp/ganda_grammar_bundle")
for folder in ['/tmp/ganda_grammar_bundle/ganda_grammar', '/tmp/ganda_grammar_bundle/ganda_grammar/rules', '/tmp/ganda_grammar_bundle/scripts']:
    Path(folder).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BUNDLE))


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/__init__.py
"""ganda-grammar: Luganda spelling and grammar correction by cited rules."""
from .rules import RULES, Edit, Result, correct

__version__ = "0.1.0"

__all__ = ["RULES", "Edit", "Result", "correct"]


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/cli.py
"""ganda-grammar correct: correct Luganda text. ganda-grammar fix: write a corrected copy of a dataset.
ganda-grammar mine: collect corrections from a corpus."""
import argparse
import json
import sys
from dataclasses import asdict
from itertools import chain


def main(argv=None):
    ap = argparse.ArgumentParser(prog="ganda-grammar", description=__doc__)
    sub = ap.add_subparsers(dest="cmd", required=True)

    c = sub.add_parser("correct", help="correct text given as an argument, or on stdin")
    c.add_argument("text", nargs="?")
    c.add_argument("--json", action="store_true", help="print the corrected text and its edits as JSON")

    f = sub.add_parser("fix", help="write a corrected copy of .txt and .jsonl files, folders, or a Hugging Face dataset")
    f.add_argument("paths", nargs="*", help=".txt and .jsonl files, or folders of them")
    f.add_argument("-o", "--out", required=True, help="folder for the corrected copies (a folder's files keep their place)")
    f.add_argument("--hf", help="also correct this Hugging Face dataset, written to OUT as JSONL")
    f.add_argument("--split", default="train")
    f.add_argument("--field", action="append", metavar="FIELD",
                   help="text field of .jsonl rows and dataset rows to correct (repeatable; default: text)")
    f.add_argument("--workers", type=int, help="processes to use (default: one per CPU)")
    f.add_argument("--log", help="also write one JSONL row per changed line or field, with its edits")

    m = sub.add_parser("mine", help="run the rules over a corpus, write one JSONL row per corrected sentence")
    m.add_argument("paths", nargs="*", help=".txt and .jsonl files, or folders of them")
    m.add_argument("--hf", help="also stream this Hugging Face dataset, e.g. henryhale/luganda-pretrain2")
    m.add_argument("--split", default="train")
    m.add_argument("--exclude-source", action="append", default=[], metavar="SOURCE",
                   help="skip dataset rows whose source column is SOURCE (repeatable)")
    m.add_argument("--field", default="text", help="text field of .jsonl rows and dataset rows (default: text)")
    m.add_argument("--limit", type=int, help="stop after this many corrected sentences")
    m.add_argument("-o", "--out", default="corrections/corrections.jsonl",
                   help="output file; keep it in corrections/, the folder that is published (default: %(default)s)")

    a = ap.parse_args(argv)
    if a.cmd == "correct":
        from . import correct

        text = sys.stdin.read() if a.text is None else a.text
        r = correct(text)
        if a.json:
            print(json.dumps({"text": r.text, "edits": [asdict(e) for e in r.edits]}, ensure_ascii=False, indent=2))
        else:
            print(r.text, end="" if a.text is None else "\n")
    elif a.cmd == "fix":
        from .fix import fix

        if not a.paths and not a.hf:
            ap.error("fix needs paths, --hf, or both")
        try:
            r = fix(a.paths, a.out, a.field or ["text"], a.hf, a.split, a.workers, a.log)
        except ValueError as e:
            ap.error(str(e))
        for rule, n in r.counts.most_common():
            print(f"{rule}\t{n}")
        print(f"{r.changed:,} of {r.seen:,} lines changed in {r.seconds:.1f}s; wrote {a.out}", file=sys.stderr)
    else:
        from .mine import hf_texts, local_texts, mine

        if not a.paths and not a.hf:
            ap.error("mine needs paths, --hf, or both")
        texts = chain(local_texts(a.paths, a.field),
                      hf_texts(a.hf, a.split, a.field, set(a.exclude_source)) if a.hf else ())
        for rule, n in mine(texts, a.out, a.limit).most_common():
            print(f"{rule}\t{n}")
        print(f"wrote {a.out}", file=sys.stderr)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/fix.py
"""Fix a dataset: write a corrected copy of every .txt and .jsonl file, or of a Hugging Face dataset.

.txt files are corrected line by line; .jsonl rows have their text fields corrected and every other field kept.
Lines and rows the rules leave alone are copied byte for byte. The input is never overwritten.

An optional log has one JSONL row per changed line or field: {"source", "line", "field", "original", "corrected",
"edits"}, with edit offsets into "original", enough to review a change or undo it.
"""
import json
import os
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from dataclasses import astuple
from pathlib import Path

from .rules import correct

CHUNK = 2000  # lines a worker corrects at a time
SUFFIXES = (".txt", ".jsonl")


def _fix_line(kind, fields, line):
    """(the line to write, [(field, original, corrected, edits)] for each changed text)."""
    if kind == ".txt":
        r = correct(line)
        return (r.text, [(None, line, r.text, [astuple(e) for e in r.edits])]) if r.edits else (line, [])
    try:
        row = json.loads(line) if line.strip() else None
    except json.JSONDecodeError:
        return line, []  # not JSON: copied as it is
    if not isinstance(row, dict):
        return line, []
    changes = []
    for f in fields:
        if isinstance(row.get(f), str):
            r = correct(row[f])
            if r.edits:
                changes.append((f, row[f], r.text, [astuple(e) for e in r.edits]))
                row[f] = r.text
    if not changes:
        return line, []
    end = "\r\n" if line.endswith("\r\n") else "\n" if line.endswith("\n") else ""
    return json.dumps(row, ensure_ascii=False) + end, changes


def _fix_chunk(args):
    kind, fields, lines = args
    return [_fix_line(kind, fields, line) for line in lines]


def _chunks(lines, kind, fields):
    chunk = []
    for line in lines:
        chunk.append(line)
        if len(chunk) == CHUNK:
            yield kind, fields, chunk
            chunk = []
    if chunk:
        yield kind, fields, chunk


def files(paths, out):
    """(input, output) for each .txt and .jsonl file; a folder's files keep their place under out."""
    out = Path(out)
    for p in map(Path, paths):
        if p.is_dir():
            for f in sorted(x for x in p.rglob("*") if x.is_file() and x.suffix in SUFFIXES):
                yield f, out / f.relative_to(p)
        elif p.suffix in SUFFIXES:
            yield p, out / p.name
        else:
            raise ValueError(f"{p}: not a .txt or .jsonl file, nor a folder")


class Fixer:
    def __init__(self, fields=("text",), workers=None, log=None):
        self.fields = tuple(fields)
        self.workers = workers or os.cpu_count() or 1
        self.counts = Counter()  # edits per rule
        self.seen = self.changed = 0
        self.log = open(log, "w", encoding="utf-8") if log else None
        self.pool = ProcessPoolExecutor(self.workers) if self.workers > 1 else None

    def close(self):
        if self.pool:
            self.pool.shutdown()
        if self.log:
            self.log.close()

    def run(self, source, lines, write, kind):
        """Correct lines (an iterable) of one source, calling write for each output line in order."""
        chunks = _chunks(lines, kind, self.fields)
        results = self.pool.map(_fix_chunk, chunks) if self.pool else map(_fix_chunk, chunks)
        n = 0
        for chunk in results:
            for line, changes in chunk:
                n += 1
                write(line)
                self.changed += bool(changes)
                for field, before, after, edits in changes:
                    self.counts.update(e[0] for e in edits)
                    if self.log:
                        self.log.write(json.dumps({
                            "source": source, "line": n, "field": field, "original": before, "corrected": after,
                            "edits": [dict(zip(("rule", "start", "end", "before", "after"), e)) for e in edits],
                        }, ensure_ascii=False) + "\n")
        self.seen += n

    def fix_file(self, src, dst):
        if dst.resolve() == src.resolve():
            raise ValueError(f"{src}: the output would overwrite the input")
        dst.parent.mkdir(parents=True, exist_ok=True)
        with src.open(encoding="utf-8", errors="replace", newline="") as fin, \
                dst.open("w", encoding="utf-8", newline="") as fout:
            self.run(str(src), fin, fout.write, src.suffix)

    def fix_hf(self, repo, out, split="train"):
        """Write a Hugging Face dataset to out as JSONL, with its text fields corrected."""
        try:
            from datasets import load_dataset
        except ImportError as e:
            raise ImportError("fixing a Hugging Face dataset needs: pip install datasets") from e
        out = Path(out)
        out.parent.mkdir(parents=True, exist_ok=True)
        rows = (json.dumps(row, ensure_ascii=False, default=str) + "\n"
                for row in load_dataset(repo, split=split, streaming=True))
        with out.open("w", encoding="utf-8") as fout:
            self.run(f"{repo}:{split}", rows, fout.write, ".jsonl")


def fix(paths, out, fields=("text",), hf=None, split="train", workers=None, log=None):
    """Write corrected copies of paths (files or folders) under out, and of the dataset hf; return the Fixer's totals."""
    fixer = Fixer(fields, workers, log)
    start = time.time()
    try:
        for src, dst in files(paths, out):
            fixer.fix_file(src, dst)
        if hf:
            fixer.fix_hf(hf, Path(out) / f"{hf.replace('/', '__')}.{split}.jsonl", split)
    finally:
        fixer.close()
    fixer.seconds = time.time() - start
    return fixer


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/mine.py
"""Mine corrections: run the rules over a corpus and write one JSONL row per corrected sentence.

Row: {"source", "original", "corrected", "edits": [{"rule", "start", "end", "before", "after"}]}, with offsets
into "original". Repeated sentences are written once.
"""
import json
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from .rules import correct
from .rules.base import sentences


def local_texts(paths, field="text"):
    """(source, text) from .txt files (whole file) and .jsonl files (each row's field); folders are searched."""
    for p in map(Path, paths):
        for f in sorted(x for x in p.rglob("*") if x.suffix in (".txt", ".jsonl")) if p.is_dir() else [p]:
            if f.suffix == ".jsonl":
                for line in f.open(encoding="utf-8"):
                    row = json.loads(line) if line.strip() else {}
                    if isinstance(row, dict) and isinstance(row.get(field), str):
                        yield str(f), row[field]
            else:
                yield str(f), f.read_text(encoding="utf-8", errors="replace")


def hf_texts(repo, split="train", field="text", exclude_sources=()):
    """(source, text) streamed from a Hugging Face dataset, skipping rows whose "source" is in exclude_sources."""
    try:
        from datasets import load_dataset
    except ImportError as e:
        raise ImportError("mining a Hugging Face dataset needs: pip install datasets") from e
    for row in load_dataset(repo, split=split, streaming=True):
        src = row.get("source")
        if src not in exclude_sources:
            yield f"{repo}:{src}" if src else repo, row[field]


def mine(texts, out, limit=None):
    """Write the corrected sentences of (source, text) pairs to out; return the edit count per rule."""
    # ponytail: one process; split the input across processes if a Kaggle session runs out of time
    seen, counts, rows = set(), Counter(), 0
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        for source, text in texts:
            for s, e in sentences(text):
                sentence = text[s:e]
                if hash(sentence) in seen:
                    continue
                r = correct(sentence)
                if not r.edits:
                    continue
                seen.add(hash(sentence))
                row = {"source": source, "original": sentence, "corrected": r.text, "edits": [asdict(x) for x in r.edits]}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                counts.update(x.rule for x in r.edits)
                rows += 1
                if rows == limit:
                    return counts
    return counts


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/__init__.py
"""Rule engine. Each family module has RULES (id → description, source) and edits(text, tokens) → [Edit]."""
from . import agreement, elision, joining, orthography
from .base import Edit, Result, apply, luganda_tokens, sentences

FAMILIES = (agreement, elision, joining, orthography)  # where two edits overlap, the earlier family's wins
RULES = {rid: rule for f in FAMILIES for rid, rule in f.RULES.items()}


def correct(text):
    kept = []
    for start, end in sentences(text):
        toks = luganda_tokens(text, start, end)
        found = [e for f in FAMILIES for e in f.edits(text, toks)]
        ours = []  # edits stay inside their sentence, so overlaps are only checked within it
        for e in found:
            if all(e.end <= k.start or e.start >= k.end for k in ours):
                ours.append(e)
        kept += sorted(ours, key=lambda e: e.start)
    return Result(apply(text, kept), kept)


__all__ = ["RULES", "Edit", "Result", "correct"]


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/agreement.py
"""Agreement: an adjective or possessive right after a noun takes the noun's class prefix.

Only after the nouns listed here, whose class is certain. A prefix alone isn't enough: eki-/obu-/ama- also start
relative verbs (ekisinga obunene) and ordinary verbs (amanyi = he knows).
"""
from .base import APOSTROPHES, Edit
from .elision import TRIGGERS

SRC = "Sternefeld & Sseguya, A Luganda Grammar (2015)"
RULES = {
    "AGREE_ADJ": ("an adjective takes its noun's class: ebintu ekirungi → ebintu ebirungi", f"{SRC}, §7"),
    "AGREE_POSS": ("a possessive takes its noun's class: ebitabo kyange → ebitabo byange", f"{SRC}, §6"),
}


def _pairs(sg, pl, stems):
    """Nouns of a singular/plural class pair: eki + tabo = ekitabo, eky + alo = ekyalo before a vowel."""
    glide = {"i": "y", "u": "w"}
    return {(pre[:-1] + glide[pre[-1]] if s[0] in "aeiou" else pre) + s: cls
            for s in stems.split() for cls, pre in (sg, pl)}


NOUNS = {
    **_pairs(("7", "eki"), ("8", "ebi"), """ntu fo biina seera tongole tundu gambo buga zibu ragiro rwadde faananyi
        binja gendererwa zimbe rabo wandiiko tabo kopo saawe bonerezo teeso rowoozo senge tone kwekweto rooto buuzo
        banja sanja rungo gere tanda muli bira go nnya kolwa ka alo uma ambalo"""),
    **_pairs(("12", "aka"), ("14", "obu"), """kiiko lulu wuka tale bonero benje tambi tabo baga kodyo faananyi piira
        sente waayiro ti solya yumba ntu papula zannyo goye veera wunga gaali wale sawo tebe buga sozi saale cupa
        pande motoka duuka kebe seera"""),
    **dict.fromkeys("""obulamu obugagga obwakabaka obukulembeze obujulizi obubaka obukuumi obusobozi obwenkanya
        obulwadde obuvunaanyizibwa""".split(), "14"),
    **dict.fromkeys("""olunaku olugendo olusozi olulimi oluggi olubuto olupapula olugoye olutalo oluyimba oluguudo
        olukiiko olusuku olususu olukalala olugero olubiri oluzzi olunyiriri olwazi""".split(), "11"),
    # Not loanwords with e + double consonant (essanduuko, essimu, ebbaluwa, ekkanisa, eggaali): they are class 9
    **dict.fromkeys("""erinnya eriiso ejjinja essomero eddoboozi ettooke eggwanga ekkubo eggye eggi etteeka essaza
        eddagala ettendekero ekkomera essomo eddwaliro""".split(), "5"),
    **dict.fromkeys("""amannya amaaso amayinja amasomero amaloboozi amatooke amawanga amakubo amagye amagi amateeka
        amasaza amalagala amatendekero amakomera amasomo amalwaliro amazzi amata amafuta amaanyi amagezi amawulire
        amaka amaziga amazima""".split(), "6"),
}

# Adjective prefixes with the augment (§7). Class 5 is left out: its forms are irregular (eddene, eddungi, eribi).
ADJ_PREFIX = {"7": "eki", "8": "ebi", "12": "aka", "14": "obu", "11": "olu", "6": "ama"}
ADJ_STEMS = "nene lungi bi kulu tono to nafu siru lalu lamu kadde gezi mpi gazi zito ddugavu myufu kalu bisi tuufu ngi pya"


def adjective(cls, stem):
    pre = ADJ_PREFIX[cls]
    return pre + ("r" + stem[1:] if pre.endswith("i") and stem[0] == "l" else stem)  # ki + lungi = kirungi


# Forms that can be wrong after a noun of another class. Not obu- forms, which are also adverbs (akola obulungi),
# nor forms that are also nouns: akatono 'a little', akabi 'danger', akakadde 'million', amakulu 'meaning',
# amato 'boats', amagezi 'wisdom'.
WRONG_ADJ = {adjective(c, s): (c, s) for c in ADJ_PREFIX for s in ADJ_STEMS.split() if c != "14"}
for w in "akatono akabi akakadde amakulu amato amagezi".split():
    del WRONG_ADJ[w]

# Possessive prefixes before the stem's a (§6). Classes 1 and 2 are left out: wange, bange are also vocatives.
POSS_PREFIX = {"3": "gw", "4": "gy", "5": "ly", "6": "g", "7": "ky", "8": "by", "9": "y", "10": "z", "11": "lw",
               "12": "k", "13": "tw", "14": "bw", "15": "kw"}


def possessive(cls, stem):
    pre = POSS_PREFIX[cls]
    return pre + (stem[1:] if stem == "aabwe" and pre[-1] in "wy" else stem)  # kyabwe but gaabwe (§1.1.6)


POSS = {POSS_PREFIX[c] + s: (c, "aabwe" if s == "abwe" else s)
        for c in POSS_PREFIX for s in ("ange", "affe", "ammwe", "aabwe", "abwe")}


def edits(text, toks):
    out = []
    for prev, a, b in zip([None] + toks, toks, toks[1:]):
        cls = NOUNS.get(a.text[0].lower() + a.text[1:])
        # Skip y'ekibuga ebirungi and bya ekibiina ebirungi: the adjective may belong to an earlier noun.
        if (not cls or a.end == b.start or text[a.end:b.start].strip(" \t")
                or (a.start and text[a.start - 1] in APOSTROPHES + "-")
                or (prev and prev.text.lower() in TRIGGERS)):
            continue
        if (hit := WRONG_ADJ.get(b.text)) and hit[0] != cls and cls in ADJ_PREFIX:
            out.append(Edit("AGREE_ADJ", b.start, b.end, b.text, adjective(cls, hit[1])))
        elif (hit := POSS.get(b.text)) and hit[0] != cls:
            out.append(Edit("AGREE_POSS", b.start, b.end, b.text, possessive(cls, hit[1])))
    return out


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/base.py
"""What every rule family shares: edits, sentences, tokens, and the check that a word or sentence is Luganda."""
import re
from dataclasses import dataclass, field
from typing import NamedTuple

APOSTROPHES = "'’ʼ‘"


@dataclass(frozen=True)
class Edit:
    rule: str  # rule id, see RULES
    start: int  # character offsets into the original text
    end: int
    before: str
    after: str


@dataclass
class Result:
    text: str
    edits: list = field(default_factory=list)


class Token(NamedTuple):
    start: int
    end: int
    text: str


def apply(text, edits):
    """Apply non-overlapping edits, sorted by start."""
    out, pos = [], 0
    for e in edits:
        out += [text[pos:e.start], e.after]
        pos = e.end
    return "".join(out) + text[pos:]


SENTENCE = re.compile(r"[^\s.!?][^.!?\n]*[.!?]*")
WORD = re.compile(r"[^\W\d_]+")


def sentences(text):
    """(start, end) of each sentence, without surrounding whitespace."""
    for m in SENTENCE.finditer(text):
        yield m.start(), m.start() + len(m[0].rstrip())


# A Luganda word is a run of syllables (consonant cluster + short or long vowel) and ends in a vowel. Only the first
# syllable may start with a vowel: two different vowels in a row (believe, milieu) aren't written in Luganda.
GEMINATE = "(?:bb|cc|dd|ff|gg|jj|kk|mm|nn|pp|ss|tt|vv|zz|ŋŋ)"
NC = "(?:m[bpfv]|n[dtkzfgjsvc])"  # nasal + consonant (Walusimbi 2015, §1.1.5, §1.1.7)
ONSET = f"(?:(?:nny|ny|{GEMINATE}|{NC}|[bcdfgjklmnprstvzŋ])[wy]?|[wy])"
VOWEL = "(?:aa|ee|ii|oo|uu|[aeiou])"
SHAPE = re.compile(f"{VOWEL}?(?:{ONSET}{VOWEL})*")

# English words with a Luganda shape that a spelling rule would change, found in henryhale/luganda-pretrain2
ENGLISH = frozenset("""
mobile believe file milieu twee failure release tile mile missile silence angelo vile delaware relative envelope
bela relate elite ila accelerate delete jubilee milo bile simile deliverance nile eliminate timeline compile militia
relieve pile consuelo lilongwe delegate reliance elaborate facilitate pele voile leila utilize tensile silica cilia
dilute vila diligence affiliate mobilize deliberate relevance pipeline delicate fossilize silicate facile revile
sweetie silo reconcile futile dilemma vigilante elusive gentile defile resilience gila felice guile relocate
vigilance automobile delude antelope baseline docile vuvuzela volatile virile filipino smile while
""".split())

# Function words of languages that share Luganda's word shape (Swahili, Chichewa, Portuguese/Spanish) and of English,
# leaving out the ones that are also Luganda words (a, be, so, we)
FOREIGN = frozenset("""
ni katika kwamba lakini hii sana pia kuwa ambao ambaye hiyo wewe mimi yeye huo hilo wao sisi nini
anthu kuti ndi ndipo ine iye que ele ela dele pela pelo para uma com los las del por
the and of is are was were to in on at for with from by this that these those it its as an or not no have has had
will would can could which who what when where there here their they them he his she her you your our also only but if
than then into about after before
""".split())


def looks_luganda(word):
    """Lowercase, made of Luganda syllables, not a known English word, and not 30+ letters of glued-together text."""
    return word.islower() and len(word) <= 30 and word not in ENGLISH and SHAPE.fullmatch(word) is not None


def luganda_tokens(text, start, end):
    """The words of text[start:end], or [] when the sentence doesn't read as Luganda, so no rule touches it."""
    # ponytail: word shape + foreign function words; add a language-ID model (fastText lid.176) if mixed text gets through
    toks = [Token(m.start(), m.end(), m[0]) for m in WORD.finditer(text, start, end)]
    # an elided word before its apostrophe (the ky of ky'okulya) isn't a word to judge
    low = [t.text for t in toks if t.text.islower() and text[t.end:t.end + 1] not in tuple(APOSTROPHES)]
    if not low or FOREIGN.intersection(low) or sum(map(looks_luganda, low)) < 0.6 * len(low):
        return []
    return toks


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/elision.py
"""Elision: a short word ending in a vowel loses it before a vowel-initial word, marked with an apostrophe."""
from .base import APOSTROPHES, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "ELISION_NA": ("na/ne before a vowel: na abantu → n'abantu", f"{SRC}, §1.1.4, §1.3(b)"),
    "ELISION_NGA": ("nga before a vowel: nga ali → ng'ali", f"{SRC}, §1.3(c)"),
    "ELISION_POSS": ("a possessive before a vowel: kya okulya → ky'okulya", f"{SRC}, §1.3(g)"),
    "ELISION_REL": ("bwe, gye in set phrases: bwe atyo → bw'atyo, gye ali → gy'ali", f"{SRC}, §1.3(j)"),
}

# wa is left out: it is also "where?" and stays whole (Ogula wa emmere?, §1.3(r))
TRIGGERS = {"na": "ELISION_NA", "ne": "ELISION_NA", "nga": "ELISION_NGA"} | dict.fromkeys(
    "kya bya lya gwa gya za lwa twa bwa kwa ka ga ya ba".split(), "ELISION_POSS")
NOT_AFTER_POSSESSIVE = {"era", "ate", "oba"}  # conjunctions: a possessive before them ends a phrase
# Only set phrases: bwe, gye, we are also possessives (obulamu bwe, omwana we ali), and those are never elided (§1.3(s))
RELATIVE = {"bwe": {"atyo", "otyo", "etyo", "ati", "oti", "eti"}, "gye": {"ali", "oli"}}


def edits(text, toks):
    out = []
    for a, b in zip(toks, toks[1:]):
        rule = TRIGGERS.get(a.text.lower()) if a.text[1:].islower() else None
        if b.text in RELATIVE.get(a.text.lower(), ()) and a.text[1:].islower():
            rule = "ELISION_REL"
        if (rule and b.text[0] in "aeiou" and len(b.text) > 1 and looks_luganda(b.text)
                and a.end < b.start and not text[a.end:b.start].strip(" \t")
                and not (a.start and text[a.start - 1] in APOSTROPHES + "-")
                and not (rule == "ELISION_POSS" and b.text in NOT_AFTER_POSSESSIVE)):
            out.append(Edit(rule, a.end - 1, b.start, text[a.end - 1:b.start], "'"))
    return out


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/joining.py
"""Word division: words the official orthography writes as one, and words it writes apart."""
from pathlib import Path

from .agreement import NOUNS
from .base import APOSTROPHES, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "JOIN_NA_PRONOUN": ("na and a pronoun are one word: na bo → nabo, na nze → nange", f"{SRC}, §1.3(a)"),
    "JOIN_FIXED": ("words written as one: buli jjo → bulijjo, mpola mpola → mpolampola, eky'okulabirako → ekyokulabirako",
                   f"{SRC}, §1.3(n), (w), (z)"),
    "SPLIT_NE": ("narrative ne is a word of its own: nebagenda → ne bagenda", f"{SRC}, §1.3(b)"),
    "SPLIT_LOC": ("mu and ku are words of their own: mukibuga → mu kibuga", f"{SRC}, §1.3(d)"),
}

NA_PRONOUN = {"nze": "nange", "ggwe": "naawe", "ye": "naye", "ffe": "naffe", "mmwe": "nammwe", "bo": "nabo",
              "kyo": "nakyo", "byo": "nabyo", "zo": "nazo", "gwo": "nagwo", "gyo": "nagyo", "lyo": "nalyo",
              "bwo": "nabwo", "lwo": "nalwo"}
TWO_WORDS = {  # §1.3(n) bulijjo; §1.3(w) a word said twice is written once
    ("buli", "jjo"): "bulijjo", ("mpola", "mpola"): "mpolampola", ("katono", "tono"): "katonotono",
    ("ddala", "ddala"): "ddaladdala", ("wonna", "wonna"): "wonnawonna", ("okukola", "kola"): "okukolaakola",
    ("okutema", "tema"): "okutemaatema", ("okubuusa", "buusa"): "okubuusabuusa", ("okudaala", "daala"): "okudaaladaala",
}
# §1.3(z) nouns: ekyokulya 'food'. After a class 7/8 word, eky'okulya means 'for eating' (ekintu eky'okulya) and stays.
COMPOUNDS = {"eky": {"okulya", "okulabirako", "okulwanyisa", "okunywa", "okwegezaamu"},
             "eby": {"okulya", "okulabirako", "okulwanyisa", "okunywa", "enjigiriza", "obulamu", "obufuzi"}}
CLASS_7_8 = ("eki", "ebi", "eky", "eby", "ki", "bi")

VERBS = frozenset(line for line in Path(__file__).with_name("verbs.txt").read_text(encoding="utf-8").splitlines()
                  if line and not line.startswith("#"))
# ne + subject prefix + verb stem. Only prefixes whose noun forms start with a or o: n'abagamba sounds nabagamba, so a
# joined nebagamba can only be ne bagamba. Not ki, bi, li, zi, gi: nebikolwa sounds the same as n'ebikolwa.
NE_PREFIXES = ("ba", "tu", "mu", "gu", "ga", "lu", "ka", "bu", "ku")
# The listed nouns without their augment, as they follow mu and ku: ekibuga → kibuga, erinnya → linnya
BARE_NOUNS = frozenset("l" + n[2:] if n.startswith("er") else n[1:] for n in NOUNS)


def _join(first, word):
    return word[0].upper() + word[1:] if first[0].isupper() else word


def edits(text, toks):
    out = []
    for prev, a, b in zip([None] + toks, toks, toks[1:]):
        gap = text[a.end:b.start]
        if not a.text[1:].islower():
            continue
        key = (a.text.lower(), b.text)
        if gap.strip(" \t") == "" and gap and not (a.start and text[a.start - 1] in APOSTROPHES + "-"):
            if key[0] == "na" and b.text in NA_PRONOUN:
                out.append(Edit("JOIN_NA_PRONOUN", a.start, b.end, text[a.start:b.end], _join(a.text, NA_PRONOUN[b.text])))
            elif key in TWO_WORDS:
                out.append(Edit("JOIN_FIXED", a.start, b.end, text[a.start:b.end], _join(a.text, TWO_WORDS[key])))
        elif (len(gap) == 1 and gap in APOSTROPHES and b.text in COMPOUNDS.get(key[0], ())
              and not (prev and not text[prev.end:a.start].strip() and prev.text.lower().startswith(CLASS_7_8))):
            out.append(Edit("JOIN_FIXED", a.end, b.start, gap, ""))
    for t in toks:
        low = t.text[0].lower() + t.text[1:]
        whole = not t.start or not (text[t.start - 1].isalnum() or text[t.start - 1] in APOSTROPHES + "-")
        if not whole or not looks_luganda(low):
            continue
        if low[:2] == "ne" and low[2:4] in NE_PREFIXES and low[4:] in VERBS:
            out.append(Edit("SPLIT_NE", t.start, t.end, t.text, f"{t.text[:2]} {t.text[2:]}"))
        elif low[:2] in ("mu", "ku") and low[2:] in BARE_NOUNS and low[2:] not in VERBS:  # mulinnya 'you climb'
            out.append(Edit("SPLIT_LOC", t.start, t.end, t.text, f"{t.text[:2]} {t.text[2:]}"))
    return out


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/orthography.py
"""Spelling rules of the official orthography, applied inside single lowercase Luganda words."""
import re
from pathlib import Path

from .base import APOSTROPHES, NC, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "ORTH_L_R": ("r, not l, after e or i: ebilala → ebirala", f"{SRC}, §1.3"),
    "ORTH_VV_NC": ("single vowel before a nasal + consonant (mb, nd, nj, nz…): ensiimbi → ensimbi", f"{SRC}, §1.1.5, §1.1.7"),
    "ORTH_VV_CW": ("single vowel after a consonant + w or y, except ggw and ggy: omukyaala → omukyala", f"{SRC}, §1.1.6"),
    "ORTH_LENGTH": ("long vowels and long consonants as the dictionary spells them: gwanga → ggwanga, olunnaku → olunaku",
                    f"{SRC}, §1.1.1-1.1.2, §1.2; spellings from the FSI Luganda Basic Course glossary (1968)"),
}
# misspelling → spelling, built by scripts/build_spellings.py
SPELLINGS = dict(line.split("\t")[:2] for line in Path(__file__).with_name("spellings.tsv").read_text(encoding="utf-8").splitlines()
                 if line and not line.startswith("#"))

PATTERNS = (
    ("ORTH_L_R", re.compile(r"(?<=[ei])l"), lambda m: "r"),
    ("ORTH_VV_NC", re.compile(rf"([aeiou])\1(?={NC})"), lambda m: m[1]),
    # Not after ny (one letter, §1.2), ggw/ggy (§1.1.6) or ngw, and not before a final -wo/-yo/-ko/-mu, which
    # lengthens the vowel of a one-syllable verb (§1.1.8: vaawo, faako).
    ("ORTH_VV_CW", re.compile(r"(?<=[bcdfgjklmnprstvzŋ][wy])(?<![gn]g[wy])(?<!ny)([aeiou])\1(?!(?:wo|yo|ko|mu)$)"),
     lambda m: m[1]),
)


def edits(text, toks):
    out = []
    for t in toks:
        after_apostrophe = t.start and text[t.start - 1] in APOSTROPHES
        if not looks_luganda(t.text) or (after_apostrophe and t.start > 1 and text[t.start - 2].isupper()):  # M'okelo
            continue
        found = [Edit(rule, t.start + m.start(), t.start + m.end(), m[0], fix(m))
                 for rule, rx, fix in PATTERNS for m in rx.finditer(t.text)]
        if not found and t.text in SPELLINGS:  # a guide pattern, when one applies, is the better explanation
            found = [Edit("ORTH_LENGTH", t.start, t.end, t.text, SPELLINGS[t.text])]
        out += found
    return out


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/spellings.tsv
# Misspellings of vowel length and doubled consonants (Walusimbi 2015, §1.1.1-1.1.2, §1.2), built by
# scripts/build_spellings.py from the FSI Luganda Basic Course glossary (1968) and corpus counts.
# misspelling	spelling	count of the misspelling	count of the spelling
gwanga	ggwanga	906	12210
olunnaku	olunaku	404	5671
nnga	nga	289	365404
lunnaku	lunaku	263	9099
egwanga	eggwanga	259	5797
omwaana	omwana	222	14500
dala	ddala	194	20843
emyaaka	emyaka	181	13236
polisi	poliisi	169	29968
myaaka	myaka	153	8775
somero	ssomero	137	4559
dwaliro	ddwaliro	89	2299
mwaana	mwana	87	5048
byaalo	byalo	84	2039
nnaye	naye	78	56625
wanno	wano	78	9221
edakiika	eddakiika	70	790
emyeezi	emyezi	69	2658
maso	maaso	66	20872
esomero	essomero	59	1613
nimiro	nnimiro	59	1614
ennyonnyi	ennyonyi	58	1076
lero	leero	58	6798
eddakika	eddakiika	55	790
enyonyi	ennyonyi	55	1076
akawungezi	akawungeezi	54	1559
kkoti	kkooti	53	17285
nyonyi	nnyonyi	46	749
ebaluwa	ebbaluwa	43	2279
edagala	eddagala	43	4156
engato	engatto	43	931
ensiimbi	ensimbi	41	6992
sebo	ssebo	41	483
kibina	kibiina	39	7101
okuda	okudda	38	5195
okuja	okujja	37	2791
dagala	ddagala	35	1624
enviri	enviiri	35	1074
amaso	amaaso	34	3671
ekibina	ekibiina	34	5715
ddakika	ddakiika	33	571
myeezi	myezi	33	1588
ddwaliiro	ddwaliro	32	2299
ebyaalo	ebyalo	32	523
emitwaalo	emitwalo	32	2240
obukade	obukadde	30	7792
ccaayi	caayi	27	342
naaye	naye	26	56625
emmeere	emmere	25	5160
mitwaalo	mitwalo	25	624
nnyonnyi	nnyonyi	24	749
omusubuzi	omusuubuzi	24	1016
kuja	kujja	23	1258
omusaanvu	omusanvu	23	1164
biffo	bifo	21	4185
kubuza	kubuuza	21	245
ngato	ngatto	21	289
mukaga	mukaaga	20	3293
ebibina	ebibiina	19	1848
edwaliro	eddwaliro	19	677
ennyanya	ennyaanya	19	317
ebikuumi	ebikumi	18	647
mmeere	mmere	18	2187
mmottoka	mmotoka	17	8621
mmotokka	mmotoka	16	8621
enimiro	ennimiro	15	758
kkuumi	kkumi	15	2352
nsiimbi	nsimbi	15	2314
olukiko	olukiiko	15	2725
emmottoka	emmotoka	14	4203
omupira	omupiira	14	2946
ssaaza	ssaza	14	1420
eterekero	etterekero	13	162
kibuuga	kibuga	13	7523
mmeza	mmeeza	13	570
musubuzi	musuubuzi	13	609
okwoogera	okwogera	13	4041
omukaga	omukaaga	13	627
buuli	buli	12	48306
eddaggala	eddagala	12	4156
kibugga	kibuga	12	7523
omuliiro	omuliro	12	4105
tuntu	ttuntu	12	366
engeeri	engeri	11	9213
ggali	ggaali	11	211
kibbi	kibi	11	1695
kutekateeka	kuteekateeka	11	170
lukiko	lukiiko	11	4064
omunyo	omunnyo	11	302
waano	wano	11	9221
eddaakiika	eddakiika	10	790
eggali	eggaali	10	268
esaza	essaza	10	538
kitunddu	kitundu	10	8891
kunaba	kunaaba	10	187
kwoogera	kwogera	10	1832
nnze	nze	10	10553
olugi	oluggi	10	649
akakade	akakadde	9	611
eddwaliiro	eddwaliro	9	677
kulowoza	kulowooza	9	644
kumannya	kumanya	9	1395
luda	ludda	9	3067
nyinimu	nnyinimu	9	108
okulaaba	okulaba	9	11717
okuleeka	okuleka	9	606
okusiinziira	okusinziira	9	4296
abakade	abakadde	8	1086
bbaaluwa	bbaluwa	8	1027
ddaggala	ddagala	8	1624
egulo	eggulo	8	2349
ekikoopo	ekikopo	8	1014
eriso	eriiso	8	528
nnyaanja	nnyanja	8	2363
obuungi	obungi	8	3178
okuwuulira	okuwulira	8	2338
omutwaalo	omutwalo	8	236
ppoliisi	poliisi	8	29968
bbuli	buli	7	48306
bibina	bibiina	7	1229
byenyanja	byennyanja	7	133
ddaakiika	ddakiika	7	571
ebifaanannyi	ebifaananyi	7	2043
ebiffo	ebifo	7	2715
emipira	emipiira	7	1390
essowani	essowaani	7	71
kikuumi	kikumi	7	1162
kubbala	kubala	7	371
kudduuka	kudduka	7	223
lugi	luggi	7	397
mmotooka	mmotoka	7	8621
okwogeera	okwogera	7	4041
oluvannyuuma	oluvannyuma	7	17949
oluvvannyuma	oluvannyuma	7	17949
ssazza	ssaza	7	1420
terekero	tterekero	7	108
ttaaba	taaba	7	98
amafuuta	amafuta	6	2470
bitabbo	bitabo	6	801
ebitabbo	ebitabo	6	1587
ekkalamu	ekkalaamu	6	68
emmeza	emmeeza	6	300
kifaanaanyi	kifaananyi	6	854
kiibiina	kibiina	6	7101
kiiki	kiki	6	2273
kulisa	kuliisa	6	72
kwogeera	kwogera	6	1832
mmaaso	maaso	6	20872
muga	mugga	6	691
ngaa	nga	6	365404
ofisi	ofiisi	6	4484
okugeenda	okugenda	6	8445
okutunuula	okutunula	6	291
okwawuula	okwawula	6	427
ooba	oba	6	48600
akawuungeezi	akawungeezi	5	1559
bbaluuwa	bbaluwa	5	1027
buluungi	bulungi	5	16085
buungi	bungi	5	2894
dirisa	ddirisa	5	224
ebituundu	ebitundu	5	4641
ekibugga	ekibuga	5	3189
ekitongoole	ekitongole	5	4751
ekolero	ekkolero	5	259
embizi	embizzi	5	626
essaaza	essaza	5	538
gulo	ggulo	5	190
kituundu	kitundu	5	8891
kutusa	kutuusa	5	252
luguuddo	luguudo	5	3409
lwokutano	lwokutaano	5	109
mambuuka	mambuka	5	158
mulwade	mulwadde	5	1022
mupira	mupiira	5	1753
ngeeri	ngeri	5	15256
nviri	nviiri	5	349
okutuunda	okutunda	5	1477
okuziimba	okuzimba	5	3278
okwagaala	okwagala	5	3739
amalalo	amalaalo	4	121
bituundu	bitundu	4	9009
buggwanjuba	bugwanjuba	4	231
bukade	bukadde	4	2701
ebifaanaanyi	ebifaananyi	4	2043
efumbiro	effumbiro	4	41
ekibuuga	ekibuga	4	3189
ekifaanaanyi	ekifaananyi	4	1648
ekifaanannyi	ekifaananyi	4	1648
ennyaanja	ennyanja	4	852
enyaanya	ennyaanya	4	317
ffee	ffe	4	4711
ggwannga	ggwanga	4	12210
kakade	kakadde	4	327
kikkumi	kikumi	4	1162
kkulya	kulya	4	1157
kulaaba	kulaba	4	2335
kummanya	kumanya	4	1395
kutuunda	kutunda	4	601
kwagaala	kwagala	4	1570
mafuuta	mafuta	4	603
mutwaalo	mutwalo	4	74
muzzadde	muzadde	4	368
nngeri	ngeri	4	15256
nyaanya	nnyaanya	4	56
obuggaali	obugaali	4	141
obunnene	obunene	4	1119
okukebeera	okukebera	4	552
okummala	okumala	4	6449
okusittula	okusitula	4	560
oluda	oludda	4	1508
omuwaala	omuwala	4	4003
wakatti	wakati	4	15060
amaawanga	amawanga	3	2417
bakade	bakadde	3	1099
bbaanga	bbanga	3	3191
bbasi	bbaasi	3	1354
biibiina	bibiina	3	1229
bitaabo	bitabo	3	801
bitunddu	bitundu	3	9009
ebbaluuwa	ebbaluwa	3	2279
ebitoongole	ebitongole	3	1935
ekikoppo	ekikopo	3	1014
ekitonngole	ekitongole	3	4751
emmotooka	emmotoka	3	4203
emyakka	emyaka	3	13236
endiiga	endiga	3	965
enjawuulo	enjawulo	3	1012
erinyo	erinnyo	3	65
kiibuga	kibuga	3	7523
kikoopo	kikopo	3	521
kuda	kudda	3	1101
kukkula	kukula	3	188
kukyussa	kukyusa	3	595
kulundda	kulunda	3	176
kuteekateka	kuteekateeka	3	170
kuwoonya	kuwonya	3	87
mmootoka	mmotoka	3	8621
munyo	munnyo	3	96
musaawo	musawo	3	564
mwakka	mwaka	3	8356
myezzi	myezi	3	1588
neda	nedda	3	784
ngga	nga	3	365404
nnimiiro	nnimiro	3	1614
nnyinnimu	nnyinimu	3	108
nsaalo	nsalo	3	932
okkubuuza	okubuuza	3	454
okubaala	okubala	3	644
okubeeraa	okubeera	3	10738
okufunna	okufuna	3	12174
okugoloola	okugolola	3	121
okukyussa	okukyusa	3	1900
okulabiika	okulabika	3	564
okumaanya	okumanya	3	5692
okunnywa	okunywa	3	1119
okussanga	okusanga	3	486
okuttunda	okutunda	3	1477
okuvuunda	okuvunda	3	165
okuyaamba	okuyamba	3	3128
olunakku	olunaku	3	5671
omuga	omugga	3	428
omusabaze	omusaabaze	3	91
omuzade	omuzadde	3	564
omweenge	omwenge	3	1544
poliissi	poliisi	3	29968
ssebbo	ssebo	3	483
tumbi	ttumbi	3	185
waakati	wakati	3	15060
wakkati	wakati	3	15060
akakaadde	akakadde	2	611
amafutaa	amafuta	2	2470
amafutta	amafuta	2	2470
amaziima	amazima	2	2053
bambii	bambi	2	381
bifaanaanyi	bifaananyi	2	454
biffaananyi	bifaananyi	2	454
ebbaaluwa	ebbaluwa	2	2279
ebiitabo	ebitabo	2	1587
ebikoopo	ebikopo	2	363
ebiteeteyi	ebiteeteeyi	2	78
ebitunddu	ebitundu	2	4641
eddaagala	eddagala	2	4156
eddagalaa	eddagala	2	4156
eekitabo	ekitabo	2	1121
eggwannga	eggwanga	2	5797
ekitaabo	ekitabo	2	1121
ekituundu	ekitundu	2	3931
ekkibiina	ekibiina	2	5715
emisannvu	emisanvu	2	205
emmotokka	emmotoka	2	4203
empeeta	empeta	2	443
endigga	endiga	2	965
engaatto	engatto	2	931
esanduuko	essanduuko	2	432
etteerekero	etterekero	2	162
ggwangga	ggwanga	2	12210
kabadda	kabada	2	46
kasirivu	kasiriivu	2	50
katonno	katono	2	2386
kawungezi	kawungeezi	2	153
kawuungeezi	kawungeezi	2	153
kibii	kibi	2	1695
kiifo	kifo	2	12593
kiirungi	kirungi	2	2839
kkunywa	kunywa	2	384
kkuyita	kuyita	2	708
kolero	kkolero	2	152
kuffuna	kufuna	2	2839
kugeenda	kugenda	2	3392
kukyaalira	kukyalira	2	69
kuleeka	kuleka	2	214
kumaanya	kumanya	2	1395
kusiinga	kusinga	2	281
kusittula	kusitula	2	181
kusituula	kusitula	2	181
kussanyusa	kusanyusa	2	196
kutaamiira	kutamiira	2	20
kutaandika	kutandika	2	1419
kutunnula	kutunula	2	202
kutuunga	kutunga	2	29
kwagalaa	kwagala	2	1570
lukkumi	lukumi	2	442
lukuumi	lukumi	2	442
lunakku	lunaku	2	9099
maserengetta	maserengeta	2	119
maziima	mazima	2	4018
mbizi	mbizzi	2	218
mmawanga	mawanga	2	3409
mmuzadde	muzadde	2	368
mpetta	mpeta	2	141
mpiiso	mpiso	2	66
mulwaadde	mulwadde	2	1022
musuubuuzi	musuubuzi	2	609
muuntu	muntu	2	12403
mwanna	mwana	2	5048
ndigga	ndiga	2	310
njawuulo	njawulo	2	1616
nkooko	nkoko	2	477
nnsalo	nsalo	2	932
nnyannja	nnyanja	2	2363
obukkadde	obukadde	2	7792
obuwaanvu	obuwanvu	2	905
okkudda	okudda	2	5195
okkugamba	okugamba	2	1346
okkulaba	okulaba	2	11717
okkumanya	okumanya	2	5692
okubaja	okubajja	2	97
okubbala	okubala	2	644
okubbeera	okubeera	2	10738
okufuuga	okufuga	2	1046
okulisa	okuliisa	2	183
okusibba	okusiba	2	626
okusirikka	okusirika	2	140
okusomesaa	okusomesa	2	1204
okussiba	okusiba	2	626
okuttonnya	okutonnya	2	232
okuttuusa	okutuusa	2	10667
okutuussa	okutuusa	2	10667
okuvvunda	okuvunda	2	165
okuwuliira	okuwulira	2	2338
okuyiita	okuyita	2	3304
okwozza	okwoza	2	201
olugguudo	oluguudo	2	1498
olukuumi	olukumi	2	147
oluvannyumma	oluvannyuma	2	17949
olwokusattu	olwokusatu	2	125
olwomukaga	olwomukaaga	2	25
olwookubiri	olwokubiri	2	229
omucceere	omuceere	2	223
omusaawo	omusawo	2	1474
omussawo	omusawo	2	1474
omusuubuuzi	omusuubuzi	2	1016
omwakka	omwaka	2	6197
ookudda	okudda	2	5195
sinnga	singa	2	8846
ttunttu	ttuntu	2	366
wakaati	wakati	2	15060
walaa	wala	2	1766


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/verbs.txt
# Verb stems (the infinitive without oku-), used to tell a verb from other words.
# Infinitives seen at least 10 times in the first 60,000 rows of henryhale/luganda-pretrain2 without tinystories-ganda,
# plus the infinitives of the English-Luganda dictionary (gandalm/resources/English-Luganda_version_2_0.txt).
# Consonant-initial stems only. Some are misspelled as written in the corpus; they only make a word count as a verb.
baaga
baagala
baagazisa
baaniriza
baawula
babala
babangula
babanja
babba
babeera
babeerera
babonereza
babonyaabonya
babudaabuda
babula
babuliira
babulira
babuulira
babuulirira
babuuza
babuzaabuza
baddiza
baddukirira
baddusa
badduukirira
bafera
bafiiriza
bafuga
bafulumya
bafumbira
bafuna
bafunira
bafuula
baga
bagaana
bagabira
bagabula
bagamba
baganya
bagasa
bagatta
bagema
bagezesa
baggalira
baggulira
baggya
bagoba
bagobaganya
bagoberera
bagula
bagulira
bagumbulula
bagumya
bagunjula
baguza
bagwa
bajja
bajjanjaba
bajjira
bajjukira
bajjukiza
baka
bakaka
bakakasa
bakaluubirira
bakaluubiriza
bakangavvula
bakebera
bakema
bakiikirira
bakiriza
bakkakkanya
bakkiriza
bakola
bakolanga
bakolera
bakoppa
bakosa
bakozesa
bakuba
bakubagiza
bakubira
bakubiriza
bakulaakulanya
bakulembera
bakunya
bakuuma
bakuumira
bakuza
bakwana
bakwasa
bakwata
bakwatira
bakyalira
bakyawa
bakyusa
bala
balaba
balabirira
balabula
balaga
balagira
balagirira
balambika
balambula
balamula
balanga
balangira
balangula
balanguza
baleeta
baleetera
baleka
balekera
balema
balemesa
balibwa
baliga
baliisa
balimba
balinnya
balirira
baliyirira
baloga
balokola
balonda
balondoola
balongoosa
baloopa
baluka
balula
baluma
balumba
balumiriza
balungamya
balwanirira
balwanyisa
balwaza
balya
bamanya
bamanyisa
bamatiza
bambaza
bamenyera
bamma
bamwa
banaaza
banga
banguka
bangula
bangulwa
banguyiza
banja
banjulira
bannyonnyola
banoonya
banoonyeza
banunula
banyaga
banyiga
banyigiriza
banyonyola
banyweza
bapimira
basaasira
basaba
basabira
basakira
basala
basalira
basaliriza
basanga
basanyusa
basasula
basasulira
basasuza
basembeza
basendasenda
basengula
basiba
basibira
basiibula
basiiga
basiima
basikiriza
basindiikiriza
basindika
basindikira
basinga
basingisa
basisinkana
basobozesa
basomera
basomesa
basonyiwa
basosola
bassa
basumulula
basunsula
basuubiza
basuula
basuulirira
basuuza
basuza
baswaza
bata
bataasa
batabaganya
batabula
batambuza
batangira
batangirira
batawaanya
bateega
bateeka
bateekateeka
bateekerateekera
bateera
bategeera
bategeeza
bategekera
batema
batendeka
batigomya
batiisa
batiisatiisa
batikka
batiza
batizibwa
batta
battira
batulugunya
batumira
batunda
batuukirira
batuusa
batuuza
batwaala
batwala
batwalira
baviira
bavuma
bavunaana
bawa
bawabula
bawagira
bawaliriza
bawamba
bawandiika
bawandiikira
bawandiisa
bawanga
bawangula
bawasa
baweebwa
baweerera
baweereza
baweerezanga
bawemula
bawola
bawolereza
bawondera
bawonya
bawulira
bawuliriza
bawuliza
bayamba
bayigga
bayigiriza
bayiira
bayimbula
bayimiriza
bayingirira
bayingiza
bayisa
bayita
baza
bazaala
bazaalira
baziika
bazikiriza
bazimba
bazimbira
bazinda
baziyiza
bazuula
bazza
bba
bbibwa
bbika
bbinkana
bbira
bbirira
bbisa
bbula
bbulula
beebaza
beefuulira
beegayirira
beekebejja
beekengera
beela
beera
beeranga
beerera
beerwa
beesiga
beetegereza
beewala
beewanika
beewuunyisa
beeyimirira
bega
bejjagala
bejjerera
bela
bembereza
bera
betenta
biddukanya
bifulumya
bifuna
bifunira
bifuula
biggya
bigoberera
bigonjoola
bigula
biika
bijja
bika
bikebera
bikira
bikirira
bikka
bikkirira
bikkiriza
bikkula
bikkulibwa
bikkulirwa
bikola
bikolanga
bikolera
bikozesa
bikula
bikulaakulanya
bikulirwa
bikuuma
bikwata
bikyusa
bimanya
binda
bindabinda
binika
bira
biraba
biraga
bireeta
bireka
birima
biriza
birizibwa
birongoosa
birwanyisa
birya
bisa
bisala
bisasulira
bisibwa
bisimba
bisiwaza
bisobozesa
bisoma
bissa
bitebya
biteeka
bitegeera
bitegeeza
bitereeza
bitereka
bitunda
bituukiriza
bituusa
bitwala
bivvuunuka
biwa
biwuliriza
biyiga
biyigiriza
biyingiza
biyita
biyonja
bizza
bobbya
boggola
bonaabona
bonabona
boneka
bonereza
bonerezebwa
bongera
bongeza
bonyaabonya
bonyaabonyezebwa
bonyabonyezebwa
booga
boogerera
bookya
boola
boolekera
botola
bowa
budaabuda
budaabudibwa
budabuda
budabudibwa
budamya
bufuna
buggya
bugiriza
bugonjoola
buguma
bugumiriza
bugumya
buguumirira
buguutanya
buka
bukala
bukola
bukozesa
bukuuma
bukwasa
bula
bulaba
bulankanya
bulira
bulirira
buliriza
bulwa
bulwanyisa
bumalira
bumba
bumbujja
bumbujjira
bumbulukuka
buna
bunya
bunyisa
bunyisibwa
bunyweza
busaasaanya
busabusa
busiiga
busoma
buteeka
butuka
butuukiriza
butuusa
butwala
buuka
buukabuuka
buukira
buulira
buuliranga
buulirira
buulirirwa
buuliriza
buulirwa
buumba
buusa
buusabuusa
buuza
buuzibwa
buvvuunuka
buwa
buyingira
buza
buzaabuza
buzaabuzibwa
buzabuza
buzibwa
buzibwabuzibwa
buzza
bwatuka
bwatukira
bwebwena
bwewala
bweyambisa
byambala
byamuka
byangatana
byekebejja
byekolera
byesonyiwa
byetegereza
byettanira
byewala
byeyambisa
byogera
caaka
cakala
cakalira
camuka
camula
camuukirira
canga
cankalana
cankalanya
cupula
daabiriza
daala
dabiriza
dda
ddaabiriza
ddaabirizibwa
ddaabulula
ddabiriza
ddanga
ddingana
ddira
ddirira
ddiriza
ddiza
ddizibwa
ddiŋŋana
dduka
ddukana
ddukannya
ddukanya
ddukanyizibwa
ddukira
ddukiranga
ddukirira
ddusa
dduukira
dduukirira
diba
dibaga
dibuuda
dibuula
dibya
digida
diibuuda
dobonkana
dobonkanya
duka
dukanya
duukirira
duula
duumira
faafaagana
faafagana
faanagana
faanana
faananyizibwa
fafaagana
fafagana
fafaganya
fanana
fananya
feebya
feffetta
fera
ffa
ffuna
fiira
fiirira
fiiriza
fiirizibwa
fiirwa
firwa
fissa
fuba
fubutuka
fudemba
fufuggaza
fuga
fuganga
fugibwa
fugira
fuka
fukamira
fukirira
fukumuka
fula
fuluma
fulumira
fulumizibwa
fulumya
fuluuta
fuma
fumba
fumbira
fumbiriganwa
fumbirwa
fumbisa
fumbiza
fumiitiriza
fumintiriza
fumita
fumitibwa
fumitiriza
funa
funanga
funda
fundikira
fundikirwa
fungiza
funibwa
funika
funira
funirwa
funyisa
funyulula
funza
futiza
futyanka
fuuka
fuukafuuka
fuukira
fuukuuka
fuula
fuulibwa
fuuna
fuuwa
fuuweeta
fuuyira
fuuyirwa
fuuyisa
fuuza
fuyira
fuyisa
gaana
gaanibwa
gaanira
gaaya
gaba
gabana
gabanibwa
gabanya
gabanyizibwa
gabibwa
gabira
gabirira
gabirwa
gabula
gabulwa
gabwa
gafuna
gafuula
gaggawala
gaggawaza
gaggya
gagula
gajambula
gajambulwa
gakima
gakola
gakozesa
gakuuma
galamira
galamiza
galamizibwa
gamba
gambibwa
gana
gandawaza
ganja
ganyula
ganyulwa
ganyurwa
ganza
gasa
gasimbagana
gassa
gata
gatambuza
gateeka
gatibwa
gatta
gattibwa
gattika
gattira
gattulula
gattululwa
gatunda
gatuusa
gatwala
gawa
gayaala
gayaalirira
gazimba
gaziwa
gaziya
gaziyizibwa
gazza
gejja
gema
gemebwa
gemesa
gemwa
genda
gendanga
gendda
gendera
genderera
gengewala
genyiwala
genza
gera
geraageranya
geraageranyizibwa
gerageranya
gereesa
gereka
geya
geza
gezebwa
gezesa
gezesebwa
gezza
ggala
ggalira
ggalirwa
ggalwa
ggibwa
ggoberera
gguka
ggula
ggulwa
ggumira
ggumiza
ggusa
ggwa
ggweera
ggya
ggyibwa
gibawa
gibba
gibwa
giddaabiriza
giddukanya
gifulumya
gifuna
gifuula
gigaana
giggala
giggya
gigoba
gigoberera
gigula
gikanika
gikebera
gikola
gikolera
gikozesa
gikuba
gikuuma
gikuwa
gikwasa
gikwata
gikyusa
gimaliriza
gimanya
gimusa
gimuwa
ginoonya
ginunula
gipangisa
gira
giraba
girabirira
giraga
gireeta
gireka
giremesa
girima
girondoola
girongoosa
girumba
girwanyisa
girya
gisaba
gisanga
gisasula
gisiba
gisiiga
gisimba
gisoma
gissa
gisuula
gitambuza
giteeka
gitegeera
gitegeeza
gitereeza
gitunda
gituukiriza
gituusa
gituwa
gitwala
givuga
giwa
giwagira
giwandiika
giwangula
giwulira
giyamba
giyingira
giyingiza
giyisa
giyita
gizimba
gizza
goba
gobaganya
gobaganyizibwa
gobebwa
gobelera
gobera
goberera
gobereranga
gobererwa
gobolola
gobwa
gogola
golokoka
golokosa
golola
gonda
gondera
gonderanga
gonjola
gonjoola
gonjoolwa
gonza
goomya
gootaana
gootaanya
gotaanya
gubawa
gufuna
gufuula
guggya
gugoba
gugulana
gugumuka
gukebera
gukola
gukozesa
gukuuma
gukwata
gula
gulaba
gulanga
guleka
gulibwa
gulira
gulirira
gulumba
gulumiza
gulumizibwa
gulwa
gulwanyisa
gulya
guma
gumaliriza
gumba
gumbulula
gumiikiriza
gumikiriza
guminkiriza
gumira
gumya
gunja
gunjula
gunywa
gusasula
gussa
gutangira
gutuusa
gutwala
guumaaza
guwa
guwangula
guwulira
guyita
guza
guzannya
guziika
guzikiza
guzza
gwa
gwekebejja
gwira
gya
gyambala
gyawula
gyekebejja
gyekenneenya
gyemera
gyetegereza
gyetooloola
gyettanira
gyeyambisa
gyibwa
gyoza
jaganya
jaguliza
jaguza
janjaba
janjabibwa
janjabwa
jeema
jeemera
jelega
jimusa
jingajinga
jingirira
jja
jjaguza
jjanga
jjanjaba
jjanjabibwa
jjanjabirira
jjanjabirwa
jjanjabwa
jjibwa
jjira
jjukira
jjukiranga
jjukirwa
jjukiza
jjukizanga
jjula
jjulira
jjulukuka
jjulula
jjumbira
jjumbirwa
jjuza
jjuzibwa
jojobya
jojolikana
jolonga
jooga
joogebwa
joonyesa
jugumira
jukira
jula
julira
juliza
julukuka
jumbira
juna
juza
kaaba
kaabira
kaabirira
kaabya
kaaniika
kaanya
kaatiriza
kaayana
kaayanira
kabala
kabasanya
kabassanya
kabawaza
kaddiwa
kafuna
kaka
kakalabya
kakana
kakanya
kakanyala
kakasa
kakasibwa
kakasiddwa
kakasiza
kakata
kakkalabya
kakkana
kakkanya
kakkasa
kakozesa
kakuba
kakuuma
kakwata
kala
kalaba
kalakata
kaliga
kalilira
kalira
kalirira
kaluba
kalubiriza
kalubya
kaluubirira
kaluubirirwa
kaluubiriza
kaluubirizibwa
kama
kamala
kambuwala
kamula
kanda
kanga
kangavula
kangavvula
kangavvulwa
kangisa
kangula
kanika
kankana
kankanya
kansa
kanyuga
kasa
kasuka
kasukira
kavvula
kaza
kazimba
kebela
kebera
keberanga
keberebwa
keberwa
kebeza
kedenza
keera
keeranga
keerewa
keesa
keewa
kekkereza
kema
kemebwa
kendeera
kendeeza
kendeezebwa
kendera
kendeza
kenena
kenenula
kenga
kenula
kerewa
ketta
kifuna
kifuula
kifuuwa
kiggumiza
kiggya
kigula
kigumira
kiika
kiikirira
kiikirirwa
kiina
kijja
kijjanya
kijjukira
kikakasa
kikiitana
kikirira
kikkaatiriza
kikkiriza
kikola
kikolera
kikolerera
kikomya
kikozesa
kikulembera
kikulira
kikuuma
kikwata
kikyusa
kiliza
kima
kimaliriza
kimanya
kimanyiira
kimuwa
kinnimba
kinnyonnyola
kinogaanya
kinogera
kinunula
kira
kiraba
kirabisa
kiraga
kirambula
kireeta
kireka
kiriza
kirizibwa
kiriziganya
kirwanyisa
kirya
kisa
kisala
kisalira
kisanga
kisasulira
kisima
kisoma
kissa
kiteeka
kitegeera
kitegeerera
kitereeza
kitunda
kitunuulira
kituukiriza
kitwala
kiwa
kiwagira
kiwakanya
kiwamba
kiwandiika
kiwandiisa
kiwangula
kiwulira
kiyiga
kiyingira
kiyisa
kiyita
kizikiriza
kizimba
kizuula
kizza
kka
kkaanya
kkaatiriza
kkakana
kkakanya
kkakkana
kkakkanya
kkalira
kkanya
kkekkereza
kkira
kkirira
kkiririza
kkiriza
kkirizibwa
kkiriziganya
kkirizisa
kkirizza
kkusa
kkuta
koboza
kogga
kola
kolagana
kolaganira
kolanga
kolebwa
koleddwa
koleeza
kolera
kolerera
kolerwa
kolima
kolimira
kolobola
kolokota
kolola
kolwa
koma
komaga
komba
komekereza
komekerezebwa
komekkereza
komekkerezebwa
komerera
komererwa
komezebwa
kommonta
komola
komolebwa
komolwa
komya
kona
konagaga
kongojja
kongoola
konjera
konkona
kontana
konziba
koobera
kookoonya
koola
koona
koonagana
koowa
koowoola
kooya
kopa
koppa
kosa
kosebwa
koseza
kotoggera
kowoola
kozesa
kozesanga
kozesebwa
kuba
kubagana
kubaganya
kubagiza
kubba
kubibwa
kubira
kubiriza
kubisa
kubudaabuda
kubuulira
kubuuza
kubwa
kuddiza
kufiiriza
kufuga
kufunira
kufuula
kugaggawaza
kugamba
kuganza
kugasa
kuggya
kugira
kugirwa
kugoba
kuguka
kuguza
kujjukira
kukakasa
kukebera
kukkiriza
kukola
kukolera
kukozesa
kukuba
kukubira
kukula
kukulembera
kukusa
kukuta
kukuuma
kukwata
kukyawa
kula
kulaakulana
kulaakulanya
kulaakulanyizibwa
kulaba
kulabirira
kulabisa
kulaga
kulakulana
kulakulanya
kuleetera
kuleka
kulema
kulembera
kulemberwa
kulembeza
kulemesa
kulira
kuliza
kulokola
kulongoosa
kulugguka
kulukusa
kulukuta
kulula
kuluma
kulumba
kulumya
kulwanyisa
kulwaza
kuma
kumaakuma
kumakuma
kumanya
kumanyisa
kumba
kumibwa
kumpanya
kunama
kunga
kungaana
kungaanira
kungaanya
kunganya
kungubaga
kungubagira
kungula
kungulwa
kunnyonnyola
kunoonya
kunta
kunya
kunyiiza
kunyumira
kunyumiza
kusa
kusaba
kusabira
kusanga
kusanyusa
kusasula
kusiba
kusinga
kusisinkana
kusobozesa
kusonyiwa
kusuula
kuta
kutaasa
kutama
kuteeka
kutegeera
kutegeeza
kutendereza
kutta
kutuka
kutula
kutulakutula
kutuukirira
kutuusa
kutwala
kutwalira
kuubagana
kuula
kuuma
kuumanga
kuumibwa
kuumira
kuumirwa
kuumwa
kuusakuusa
kuuta
kuutira
kuvuma
kuwa
kuwandiikira
kuweereza
kuwonya
kuwulira
kuwuliriza
kuyamba
kuyingira
kuyisa
kuyita
kuyunga
kuza
kuzibuwalira
kuzibwa
kuzikiriza
kuziyiza
kuŋŋaana
kuŋŋaanya
kwaana
kwaata
kwaatagana
kwagala
kwajja
kwana
kwanaganya
kwasa
kwasaganya
kwasibwa
kwasisa
kwasiza
kwata
kwatagana
kwataganira
kwataganya
kwatanga
kwatibwa
kwatira
kwatirira
kwatirwa
kwattibwa
kwebaza
kweeka
kweka
kwekeeta
kwekengera
kwekula
kwesiga
kwetaagisa
kwewuunyisa
kwongera
kya
kyaala
kyaawa
kyagenda
kyakira
kyala
kyalira
kyama
kyambala
kyamuka
kyamula
kyamya
kyankalana
kyankalanya
kyawa
kyawagana
kyayibwa
kyayisa
kyaza
kyebungulula
kyekenneenya
kyerabira
kyetegereza
kyetooloola
kyewala
kyima
kyogera
kyoleka
kyuka
kyukakyuka
kyukira
kyupula
kyusa
kyusakyusa
kyusibwa
kyusiza
kyuuka
kyuusa
laajana
laajanira
laakulana
laakulanya
laalika
laama
laawa
laaza
laba
labagana
labanga
labankana
labibwa
labika
labikira
labikirwa
labilira
labira
labirira
labiriranga
labirirwa
labiriza
labirwa
labisa
labisibwa
labula
labulwa
labwa
lafuubana
laga
lagaana
lagajjalira
lagibwa
lagira
lagirira
lagirwa
lagula
lagulwa
lakira
lakulana
lakulanya
lala
lalukira
lamaga
lamba
lambaala
lambika
lambikibwa
lambula
lambulanga
lambulula
lambuula
lambuza
lamula
lamulwa
lamusa
lamuza
landa
langa
langibwa
langira
langirira
langirirwa
lanya
lasa
lawuna
layira
layiza
layizibwa
lebeesa
lebeeta
lebera
lebeza
leega
leekaana
leekaanira
leeta
leetanga
leetebwa
leetera
lega
legejja
legesa
leguula
leka
lekaana
lekanga
lekanira
lekebwa
lekera
lekulira
lema
lemagana
lemala
lemba
lembeka
lembera
lemera
lemererwa
lemesa
lemesebwa
lemeza
lemwa
lengera
lengezza
lera
leta
letebwa
letera
libba
liddukanya
lifuna
liggya
ligula
liguza
liibwa
liikiriza
liira
liisa
likola
likozesa
likuba
likulaakulanya
likuuma
lilanya
lima
limba
limbalimba
limbibwa
limibwa
limira
limirira
limwa
lina
linaana
linda
lindanga
lindirira
linga
lingiza
linnya
linnyira
linnyirira
linnyirirwa
linnyisa
linnyisibwa
linya
linyirira
linyisa
linyisibwa
lippa
lira
liraana
lirana
lissa
liteeka
litegeera
litunda
litwala
liwa
liyirira
liyirirwa
liyita
lizza
loga
loka
lokoka
lokola
lokolebwa
lomba
londa
londebwa
londera
londerera
londesa
londola
londoola
londoolwa
longooka
longoosa
longoosebwa
longosa
longosebwa
lonkoma
loonda
loondola
loopa
loota
lowooleza
lowooza
lowoozanga
lowoozesa
lowoza
loza
lufuula
luka
lukola
lukozesa
lukuba
luluka
lulunkanira
luma
lumba
lumbagana
lumbibwa
lumika
lumirirwa
lumiriza
lumirizibwa
lumirwa
lumizibwa
lumwa
lumya
lunda
lundibwa
lundira
lungamizibwa
lungamya
lungiya
lusoma
lutwala
luubirira
luutinga
luŋŋamya
lwa
lwaala
lwaana
lwaanirira
lwaanyisa
lwala
lwalalwala
lwana
lwanagana
lwanira
lwanirira
lwanisa
lwanyisa
lwanyisibwa
lwaza
lya
lyamuka
lyanga
lyowa
maamira
magamaga
mala
malira
malirira
maliriza
malirizibwa
mannya
mannyikwa
mansa
manya
manyagana
manyanga
manyibwa
manyiddwa
manyiira
manyiiza
manyika
manyira
manyisa
manyisibwa
masamasa
matira
matiza
matizibwa
mbudaabuda
mbuulira
mbuuza
megenya
megga
meggana
meketa
menya
menyebwa
menyeka
menyekamenyeka
mera
meremeenya
meruka
merusa
meza
mila
mira
mma
mmanya
mmuka
mpa
mpanga
mpasa
mpeerera
mpeereza
mpisa
mpita
mpolereza
mponya
mpulira
mpuliriza
mubanja
mubatiza
mubba
mubbira
mubeera
mubikira
muboggolera
mubonereza
mubudaabuda
mubuulira
mubuulirira
mubuuza
mubuzaabuza
mucamula
muddira
muddiza
muddusa
mudduukirira
mufaanana
mufera
mufiiriza
mufulumya
mufumbirwa
mufumbiza
mufumita
mufuna
mufunira
mufunyisa
mufuula
mufuuwa
mufuuyira
mugaana
mugajambula
mugamba
muganza
mugatta
mugeraageranya
mugezesa
muggalira
muggulira
muggya
mugoba
mugobaganya
mugobera
mugoberera
mugondera
mugula
mugulira
mugulumiza
mugumya
muguza
mugwa
mugya
mujanjaba
mujeemera
mujja
mujjanjaba
mujjanjabira
mujjukira
mujjukiza
mukaka
mukakasa
mukaluubirira
mukambuwalira
mukanda
mukangavvula
mukansa
mukebera
mukema
mukiikirira
mukima
mukka
mukkakkanya
mukkiriza
mukkirizisa
mukola
mukolera
mukomerera
mukoppa
mukosa
mukozesa
mukuba
mukubagiza
mukubanga
mukubira
mukubiriza
mukugira
mukulisa
mukungubagira
mukungula
mukunya
mukuuma
mukuumira
mukuza
mukwana
mukwasa
mukwata
mukwatira
mukwatirira
mukyalira
mukyawa
mukyusa
mulaba
mulabirira
mulabula
mulaga
mulagira
mulamusa
mulanga
mulangira
mulangirira
muleeta
muleetera
muleka
mulekera
mulemesa
mulengezza
muliisa
mulima
mulinda
mulinnya
muliraana
mulisa
muliyirira
muloga
mulokola
mulonda
mulondoola
mulongoosa
muloopa
muloopera
muluma
mulumba
mulumiriza
mulumya
mulungula
mulwanyisa
mulya
mumalira
mumanya
mumatiza
mumenya
mumiza
mumma
mumyuka
munaaza
munafuya
munenya
munnyonnyola
munona
munoonya
munoonyeza
mununula
munyaga
munyiga
munyiiza
munyooma
munyumira
munywa
munywegera
munyweza
mupangisa
muperereza
musaasira
musaba
musabira
musala
musalira
musamba
musanga
musanyusa
musasula
musasulira
musekerera
musemberera
musiba
musibira
musibya
musigulira
musiibula
musiiga
musiima
musika
musikira
musikiriza
musimba
musimbira
musindiikiriza
musindika
musindikira
musinga
musingisa
musinza
musisinkana
musitula
musobola
musobozesa
musojja
musomera
musomesa
musonseka
musonyiwa
musoya
mussa
musunguwaza
musuubiza
musuula
musuuza
musuza
muswaza
muta
mutaasa
mutabukira
mutabula
mutambuza
mutangira
muteebereza
muteega
muteeka
muteekateeka
mutega
mutegeera
mutegeeza
mutegekera
mutema
mutemula
mutendeka
mutendereza
mutiisa
mutiisatiisa
mutikka
mutolosa
mutomera
mutongoza
mutta
muttira
mutuga
mutulugunya
mutuma
mutunda
mutunuulira
mutuuka
mutuukirira
mutuuma
mutuusa
mutuuza
mutwaala
mutwala
mutwalira
mutya
muviira
muvuga
muvuganya
muvuma
muvunaana
muwa
muwaabira
muwabula
muwagira
muwakanya
muwamba
muwandiikira
muwangula
muwasa
muwawaabira
muweebwa
muweerera
muweereza
muweerezanga
muweesa
muwemula
muwerekera
muwereza
muwola
muwolereza
muwondera
muwonya
muwooyawooya
muwozesa
muwujja
muwulira
muwuliriza
muwuliza
muwummuza
muyamba
muyigga
muyiggira
muyigiriza
muyiira
muyimbira
muyimbula
muyimiriza
muyingiza
muyisa
muyita
muyonsa
muyoolayoola
muyozaayoza
muzaala
muzaalira
muziika
muzikiriza
muzimba
muzimbira
muziyiza
muzuukiza
muzuukusa
muzuula
muzza
mwa
mwabulira
mwagala
mwagalira
mwagazisa
mwambaza
mwambula
mwanguyira
mwanika
mwaniriza
mwanjula
mwanjulira
mwanukula
mwawula
mwaza
mwebaza
mwebazanga
mwebwa
mwefuulira
mwegayirira
mwejjeereza
mwekebejja
mwekengera
mwenya
mweraliikiriza
mwesembereza
mwesiga
mwesonyiwa
mwetaagira
mwetegereza
mwetondera
mwetooloola
mwewala
mwewuunyisa
mweyabiza
mweyambisa
mweyimirira
mwogerera
mwogereza
mwogoloza
mwokya
mwolekera
mwongera
mwononekera
mwonoona
mwonoonekera
mwonoonera
myansa
myuka
myumyula
naaba
naabuuka
naatera
naaza
naazibwa
naba
nafuwa
nafuya
nakuwala
nakuwalira
nanagira
ndaba
ndabirira
ndaga
ndeetera
ndeka
ndiisa
ndiyirira
nemesa
nenya
nenyezebwa
nfumbirwa
nfunira
nfunyisa
nfuula
ngaana
ngamba
ngoba
ngobaganya
ngoberera
ngula
njagala
njigiriza
njooga
nkakasa
nkebera
nkola
nkolera
nkozesa
nkuba
nkubira
nkuuma
nkwana
nkwata
nkyalira
nneebaza
nneeraliikiriza
nneewuunyisa
nnemesa
nnoga
nnongoosa
nnoonya
nnuma
nnumba
nnumiriza
nnumya
nnunula
nnyamba
nnyanjula
nnyiiza
nnyika
nnyikira
nnyikiza
nnyogoga
nnyonnyoka
nnyonnyola
nnyonnyolwa
nnyonyoka
nnyonyola
nnyuka
noba
noga
nogera
nogerwa
nona
nonooza
nonya
nonyereza
nonyeza
noonya
noonyereza
noonyerezza
noonyeza
noonyezebwa
nooyereza
nsaba
nsalira
nsanga
nsanyusa
nsasula
nsiba
nsinga
nsinza
nsisinkana
nsomesa
nsonyiwa
nsunguwaza
nsuula
nsuza
nta
ntaasa
nteeka
ntegeeza
ntengula
ntiisatiisa
ntulugunya
ntuma
ntunuulira
ntuusa
ntwala
nunula
nunulibwa
nunulwa
nuuma
nuuna
nuusa
nvuma
nyaga
nyagibwa
nyagulula
nyakula
nyamba
nyeenya
nyenya
nyiga
nyigibwa
nyigiriza
nyigirizibwa
nyiiga
nyiigira
nyiikira
nyiikirira
nyiiza
nyika
nyikira
nyikiza
nyirira
nyiriza
nyiza
nyoga
nyogoga
nyolwa
nyomoola
nyonnyoka
nyonnyola
nyonyogera
nyonyola
nyooka
nyooma
nyoomebwa
nyuka
nyuma
nyumira
nyumirwa
nyumisa
nyumiza
nyumunguza
nyumya
nywa
nywanga
nyweeza
nywegera
nywera
nywerera
nywesa
nyweza
nywezebwa
nzaala
nzigya
nziika
nzijanjaba
nzijira
nzikiriza
nzimbira
nziruukirira
nzisa
nzita
nzitira
nziza
paakinga
pakasa
pakira
pamga
panga
pangisa
pangisibwa
papa
peeka
pepeya
perereza
pima
pimibwa
pimira
pondooka
punta
pya
ranbura
saaba
saabala
saabaza
saabulula
saaga
saagira
saala
saalimbira
saalira
saanga
saanikira
saanuuka
saasaana
saasaanira
saasaanya
saasaanyizibwa
saasana
saasanya
saasira
saasirwa
saawa
saaza
saba
sabanga
sabibwa
sabira
sabiriza
sabirwa
sadaaka
saddaaka
saddaakibwa
sagaasagana
saggula
sajjalaata
sajjuka
sajjula
saka
sakira
sala
salaasala
salanga
salibwa
salimbira
salira
salirwa
salwa
samba
sambira
samira
sanga
sangibwa
sangula
sanikira
sanirira
sanjaga
sannyalala
sannyalaza
sanukula
sanyalala
sanyalaza
sanyuka
sanyukira
sanyusa
sanyusibwa
sasaana
sasaanira
sasaanya
sasamaza
sasana
sasanya
sasika
sasira
sasula
sasulanga
sasulibwa
sasulira
sasulirwa
sasulwa
sasuza
sattira
sattiza
sattulula
sawuka
sawula
saza
seebengerera
seembera
seera
seerera
seetuka
seetula
seeyeya
segeera
seka
sekerera
sekererwa
seketerera
sekula
semba
sembera
semberera
sembereza
sembeza
sembezebwa
sena
senda
sendasenda
senga
sengejja
sengeka
senguka
sengukira
sengula
sengulira
sengulwa
sensera
senserwa
senvula
senya
sereba
sereka
serengeta
sesebbula
sesema
seyeeya
siba
sibibwa
sibira
sibirira
sibirwa
sibisa
sibuka
sibulukuka
sibwa
siga
sigala
sigalira
sigaza
sigukulula
sigula
siguukulula
siiba
siibula
siibulukuka
siiga
siigibwa
siika
siikiriza
siima
siimba
siimbula
siimibwa
siimuula
siinga
siitaana
siiwa
siiwuuka
siiyibwa
sika
sikibwa
sikira
sikiriza
sikirizibwa
sikiza
sima
simattuka
simatuka
simba
simbibwa
simbira
simbuka
simbula
simbuliza
simibwa
simira
simula
simuula
simwa
sinda
sindana
sindiikiriza
sindiikirizibwa
sindika
sindikibwa
sindikira
sindikiriza
sindirira
sindogoma
sinduukirirwa
singa
singana
singibwa
singira
singisa
singisibwa
singiza
sinza
sinzanga
sinzibwa
sinziira
sinzila
sinzira
sinziza
siramuka
siriikirira
siriira
siriiza
sirika
sirikira
sirikirwa
sirisa
sirisibwa
siruwaza
sisimuka
sisinkana
sisinkanira
sitama
situka
situkira
situla
situlanga
situnkana
situula
siwuuka
siyagguka
soba
soberwa
soboka
sobola
soboola
sobozesa
sobya
sogola
sojjolimba
soka
sokera
sokoola
solooza
soma
somanga
somba
somebwa
somera
somerera
somerwa
somesa
somesebwa
somoka
somooza
somoozebwa
somosa
somozebwa
somwa
sonda
sondera
songa
songosa
sonjola
sonsa
sonseka
sonyibwa
sonyiwa
sonyiwagana
sonyiyibwa
sonyiyizibwa
sooba
sooberera
soobola
sooka
sookanga
sookera
sooma
soomooza
soomoozebwa
soomozebwa
soona
soosowaza
soosowazibwa
soowagana
sosola
sosolagana
sosolebwa
sosolwa
sosonkereza
sosowaza
ssa
ssibwa
ssuuka
subula
subwa
sukka
sukkirira
sukkuluma
sula
sumagira
sumba
sumbuwa
sumika
sumulukuka
sumulula
sumululwa
suna
sunda
sunguwala
sunguwalira
sunguwaza
sunsula
sunsulwa
susa
sussa
suubiira
suubira
suubirwa
suubiza
suubula
suubulagana
suubuza
suuka
suula
suulibwa
suulira
suulirira
suulumba
suumusa
suumusibwa
suumuusibwa
suusuuta
suuta
suuza
suza
suzibwa
swaakira
swaaza
swakira
swala
swankula
swanyuula
swaza
swazibwa
taakiriza
taama
taasa
taasibwa
taataaganya
taataaganyizibwa
taataganya
taataganyizibwa
taawulula
taayiza
taba
tabaala
tabagana
tabaganya
tabanguka
tabangula
tabika
tabuka
tabukira
tabula
tabulatabula
tabulwa
tagala
tagambika
taganjula
taggulula
tajjulukuka
takoma
takula
talaaga
talagga
talantuka
talina
tambula
tambulanga
tambulatambula
tambulira
tambuliranga
tambuliza
tambulizibwa
tambuza
tambuzibwa
tamiira
tamiiza
tamwa
tandiika
tandiikira
tandika
tandikibwa
tandikira
tandisa
tanga
tangaala
tangaaza
tangaazibwa
tangaza
tangira
tangirira
tangirwa
tankana
tankanibwa
tanula
tanzibwa
taputa
tasa
tataaganya
tataaganyizibwa
tataganya
tataganyizibwa
tatana
tawaanya
tawaanyizibwa
tawulula
tayirira
tebeereza
tebenkera
tebenkeza
tebenta
tebentesa
tebereza
tebuka
teeba
teebera
teebereza
teeberezebwa
teebwa
teega
teegera
teeka
teekateeka
teekebwa
teekebwateekebwa
teekera
teekerateekera
teekesa
teela
teesa
teeseganya
teeseza
tega
tegana
teganya
tegeela
tegeera
tegeeragana
tegeerebwa
tegeerekeka
tegeerera
tegeerwa
tegeeza
tegeezanga
tegeezebwa
tegeka
tegekebwa
tegekeddwa
tegekera
tegera
tegeza
teka
tekateeka
tekebwa
tekesa
tema
temaatema
temagana
tematema
temula
temulwa
temya
tenda
tendeka
tendekebwa
tendekera
tendereza
tenderezanga
tenderezebwa
tendewalirwa
tengejja
tengula
terebula
tereera
tereeza
tereezebwa
tereka
terekebwa
terekera
terera
tereza
teseganya
tetenkanya
tibwa
tigomya
tiibwa
tiisa
tiisatiisa
tiisibwa
tiisibwatiisibwa
tiitiibya
tijja
tikirwa
tikka
tikkira
tikkirwa
tikkula
timba
timbibwa
timpula
tindigga
tindira
tinta
tirimbula
tisa
tisatiisa
toba
tobeka
toberera
tobya
tojjera
tokosa
tokota
toloka
tomeggana
tomera
tomeragana
tomeregana
tomerwa
tona
tonda
tondebwa
tondeka
tondoka
tonera
tongoza
tongozebwa
tonnya
tontoma
tonya
toola
tottola
tta
ttaanya
ttattana
ttibwa
ttika
ttikirwa
ttira
ttottola
ttukiza
ttulula
ttunka
ttunkana
ttunkira
tubeerera
tubira
tubudaabuda
tubuulira
tubuuza
tubuzaabuza
tuddiza
tudduukirira
tufiirira
tufuga
tufunira
tufuula
tuga
tugamba
tuganyula
tugatta
tuggya
tuguza
tujja
tuka
tukakasa
tukira
tukirira
tukiriza
tukkiriza
tukola
tukolera
tukuba
tukubira
tukubiriza
tukula
tukulembera
tukuuma
tukuumira
tukuza
tukuzibwa
tukwata
tula
tulaba
tulabirira
tulabula
tulaga
tuleetera
tuleka
tulemesa
tuliisa
tulisa
tulokola
tulongoosa
tulugunya
tulugunyizibwa
tulukuka
tuluma
tulumba
tuma
tumbuka
tumbula
tumbulwa
tumibwa
tumya
tunda
tundibwa
tundira
tundula
tunga
tunka
tunnyonnyola
tuntuza
tunula
tunulira
tununula
tunuulira
tunuulirwa
tunuza
tusa
tusaasira
tusaba
tusabira
tusasula
tusinga
tusisinkana
tusobozesa
tusomesa
tusonyiwa
tutaasa
tutabula
tutegeeza
tutendeka
tutiisatiisa
tutta
tutulugunya
tutumuka
tutumula
tutuusa
tutwala
tuuka
tuukana
tuukanya
tuukira
tuukirira
tuukirirwa
tuukiriza
tuukirizibwa
tuula
tuulanga
tuulira
tuuma
tuusa
tuusibwa
tuusiza
tuusizza
tuuyana
tuuza
tuuzanga
tuuzibwa
tuwa
tuweereza
tuwonya
tuwuliriza
tuyamba
tuyigiriza
tuyita
twaala
twaalira
twaaliza
twagala
twala
twalanga
twalibwa
twalira
twalirizibwa
twaliza
twanguyira
twawula
twetaagisa
twewuunyisa
tya
tyaba
tyanga
tyobola
tyoboola
tyoboolebwa
viira
viiraddala
vira
voola
vuba
vubiriza
vubuka
vuga
vugannya
vuganya
vuganyiza
vuganyizibwa
vugira
vugirira
vugisa
vujjirira
vuluga
vuma
vumaganya
vumbeera
vumbula
vumibwa
vumirira
vumula
vunaana
vunaanibwa
vunaanyizibwa
vunama
vunanibwa
vunanyizibwa
vunda
vundira
vunnama
vunnamira
vunuka
vunza
vuunama
vuunika
vuunula
vuvuuma
vuyisa
vva
vvoola
vvunuka
vvunula
vvuunuka
vvuunula
vvuunulwa
yaayaana
yaayaanira
yamba
yambagana
yambanga
yambibwa
yiga
yigga
yigganya
yigganyizibwa
yiggibwa
yiggira
yigira
yigiriza
yigirizibwa
yiibwa
yiika
yiikuula
yiira
yiiya
yiiyiza
yima
yimba
yimbibwa
yimbira
yimbula
yimbulwa
yimira
yimirira
yimiriza
yimirizibwa
yimuka
yimusa
yimusibwa
yinaayina
yindira
yinga
yingira
yingirira
yingiza
yingizibwa
yinza
yiringisa
yisa
yisibwa
yita
yitanga
yitibwa
yitibwanga
yitimuka
yitimusa
yitingana
yitira
yitirira
yitiriza
yitta
yiwa
yiwayiwa
yomba
yombagana
yombayomba
yombera
yongobera
yongobeza
yonja
yonka
yonsa
yoola
yooyoota
yoya
yoyoota
yozaayoza
yozayoza
yubuka
yuguuma
yulika
yunga
yungibwa
yungula
yunja
yuuga
yuuguuma
yuwa
yuza
yuzayuza
zaala
zaalibwa
zaalira
zaalisa
zaama
zaanika
zaawula
zaaza
zala
zalibwa
zambala
zannya
zannyibwa
zannyira
zannyirwa
zannyisa
zanya
zanyibwa
zanyira
zeetegereza
zeewala
zeewola
zeeyambisa
zettanira
ziba
zibawa
zibba
zibikira
zibikula
zibira
zibiriza
zibuka
zibukula
zibula
zibuwaza
zibwa
zibya
ziddaabiriza
zifukirira
zifuna
zifuula
zifuuyira
ziga
zigaba
zigabira
zigema
ziggya
zigonjoola
zigula
ziguza
ziika
ziikibwa
ziikula
ziiyiza
zijjanjaba
zika
zikira
zikirira
zikiriza
zikirizibwa
zikiza
zikola
zikozesa
zikuba
zikuuma
zikuumira
zikwasa
zikwata
zikyusa
zimanya
zimba
zimbibwa
zimbiddwa
zimbira
zimbirwa
zimbulukuka
zimbulukusa
zimbwa
zimenya
zimpa
zimuddiza
zimusasula
zimuwa
zina
zinda
zinga
zingama
zingiza
zingulula
zinira
zinoga
zinoonya
zipangisa
zira
ziraba
zirabirira
zireeta
ziriisa
zirika
zirima
zirondoola
zirongoosa
zirunda
zirwanyisa
zirya
zisa
zisaasaanya
zisala
zisanga
zisasula
zisiba
zisimba
zisindika
zisobozesa
zisoma
zissa
zitambuza
ziteeka
zitegeera
zitegeka
zitereka
zitowa
zitta
zitunda
zituusa
zitwala
ziwa
ziwandiisa
ziwangula
ziweza
ziyamba
ziyira
ziyisa
ziyita
ziyiza
ziyizibwa
zizza
zooza
zula
zuuka
zuukira
zuukiza
zuukizibwa
zuukuka
zuukusa
zuula
zuulibwa
zuulira
zuulwa
zza
zzibwa
ŋŋamba


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/scripts/build_spellings.py
"""Build src/rules/spellings.tsv: misspellings of vowel length and doubled consonants, and their dictionary spelling.

    pdftotext -layout -f 364 -l 385 "FSI - Luganda Basic Course - Instructor and Student Text.pdf" glossary.txt
    python scripts/build_spellings.py glossary.txt English-Luganda_version_2_0.txt corpus.jsonl [...] > src/rules/spellings.tsv

1. The FSI Luganda Basic Course glossary (1968) gives each word in the standard spelling: long vowels and long
   consonants written double. Entries damaged by OCR are repaired when exactly one corpus word fits.
2. Each glossary word, its plural and its bare form (without the augment) is a correct spelling.
3. A misspelling is a correct spelling with one vowel or consonant lengthened or shortened, seen in the corpus at least
   twice and at least ten times less often, and not a word of its own: not in the English-Luganda dictionary, not a
   listed noun or verb, not English, not a minimal pair from the orthography guide, not in REVIEWED_OUT.
"""
import collections
import json
import re
import sys
from pathlib import Path

from ganda_grammar.rules.agreement import NOUNS  # the package installed: pip install -e .
from ganda_grammar.rules.joining import VERBS

# Both spellings are words (Walusimbi 2015, §1.1.1-1.1.2, §1.3): never correct one into the other
MINIMAL_PAIRS = "lega leega fuka fuuka siba siiba sima siima mala maala sera seera kola koola ekisero ekiseera".split()
# Words the guide spells that a variant could hit: the interrogatives of §1.3(r) (baluwa 'which ones', not bbaluwa 'letter')
GUIDE_WORDS = "kiruwa ziruwa biruwa aluwa baluwa galuwa".split()
# Found by reading the table: a word of its own, or not sure
REVIEWED_OUT = set("""ddaala ggye omuka obba okuweera okugala okusibula buuliri obuuliri okumaala muuka waala amakati
    entebbe ntebbe kusibula kubera emere""".split())
WORD = re.compile(r"[a-zŋ']+")
TOKEN = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")  # as the rules tokenize, keeping elided words whole


def glossary(path, counts):
    """(word, class or perfect form, English meaning) from the FSI glossary text."""
    head = re.compile(r"^\s*(\(?[^\s(]+\)?)(?:\s*\(([^)]*)\))?\s{2,}(\S.*)$")
    by_len = collections.defaultdict(list)
    for w, n in counts.items():
        if n >= 3:
            by_len[len(w)].append(w)
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        m = head.match(line)
        if not m or "LUGANDA" in line or m[1].lstrip("(").startswith("."):  # page headers, bound stems (.tegeevu)
            continue
        h = m[1].strip("()\"'·/").replace("·", "").replace("''", "")
        h = re.sub(r"^0", "o", h).replace("1", "l").lower().replace(".", "")
        if re.fullmatch(r"[a-zŋ']+", h):  # clean; OCR reads mw as 'row'
            if "row" in h and counts[h.replace("row", "mw")] > counts[h]:
                h = h.replace("row", "mw")
        elif re.search(r"[a-zŋ]{3}", h):  # damaged: junk is a wildcard; keep the repair only when one word fits
            pat = re.compile("".join(re.escape(ch) if WORD.fullmatch(ch) else "[a-zŋ]{0,2}" for ch in h)
                             .replace("row", "(?:mw|row)"))
            fits = sorted(((counts[w], w) for n in range(len(h) - 3, len(h) + 4) for w in by_len[n] if pat.fullmatch(w)),
                          reverse=True)
            if not fits or (len(fits) > 1 and fits[0][0] < 5 * fits[1][0]):
                continue
            h = fits[0][1]
        else:
            continue
        out.append((h, m[2] or "", re.split(r"\s{2,}", m[3])[0].strip()))
    return out


def plural(word, cls):
    """The plural where the FSI class makes it regular (ekyalo KI-BI → ebyalo), else None."""
    for sg, pl, k in (("eki", "ebi", "KI-BI"), ("eky", "eby", "KI-BI"), ("omu", "emi", "MU-MI"), ("omw", "emy", "MU-MI"),
                      ("aka", "obu", "KA-BU")):
        if cls.startswith(k) and word.startswith(sg):
            return pl + word[3:]


def forms(word, cls):
    """The word, its plural where the class makes it regular, and its bare form."""
    out = {word, plural(word, cls)} - {None}
    return out | {w[1:] for w in out if len(w) > 4 and w[0] in "aeo" and w[1] not in "aeiou"}


def variants(w):
    """w with one vowel or consonant made long or short (l, r, w, y are never doubled, §1.2)."""
    out, i = set(), 0
    while i < len(w):
        if w[i].isalpha() and w[i] not in "lrwy":
            if i + 1 < len(w) and w[i + 1] == w[i]:
                out.add(w[:i] + w[i + 1:])
                i += 2
                continue
            out.add(w[:i] + w[i] + w[i:])
        i += 1
    return out


def main(glossary_txt, dictionary_txt, *corpus):
    counts = collections.Counter()
    for path in corpus:
        for line in open(path, encoding="utf-8"):
            counts.update(TOKEN.findall(json.loads(line)["text"]))
    correct = {}
    for word, cls, _ in glossary(glossary_txt, counts):
        for f in forms(word, cls):
            correct[f] = word
    english = {w.strip().lower() for w in open("/usr/share/dict/words") if w.strip().isalpha()}
    dic = set(WORD.findall(Path(dictionary_txt).read_text(encoding="utf-8").lower())) - english
    real = (set(correct) | set(NOUNS) | VERBS | {"oku" + v for v in VERBS} | dic | {w[1:] for w in dic if w[:1] in "aeo"}
            | set(MINIMAL_PAIRS) | {"oku" + w for w in MINIMAL_PAIRS} | set(GUIDE_WORDS))
    rows = sorted({(v, g, counts[v], counts[g]) for g in correct if counts[g] >= 20 for v in variants(g)
                   if counts[v] >= 2 and counts[g] >= 10 * counts[v] and len(v) >= 4
                   and v not in real and v not in english and v not in REVIEWED_OUT}, key=lambda r: (-r[2], r[0]))
    print("# Misspellings of vowel length and doubled consonants (Walusimbi 2015, §1.1.1-1.1.2, §1.2), built by")
    print("# scripts/build_spellings.py from the FSI Luganda Basic Course glossary (1968) and corpus counts.")
    print("# misspelling\tspelling\tcount of the misspelling\tcount of the spelling")
    for v, g, nv, ng in rows:
        print(f"{v}\t{g}\t{nv}\t{ng}")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/scripts/bundle_notebook.py
"""Write a self-contained copy of a notebook, carrying ganda-grammar in its first cells.

    python scripts/bundle_notebook.py notebooks/build_dictionary.ipynb
    → notebooks/build_dictionary_selfcontained.ipynb (or name the output as a second argument)

The copy starts with one %%writefile cell per file of the package (src/, written as ganda_grammar/, with its data files)
and of scripts/, all under BUNDLE, which goes first on sys.path. Its own cells then find BUNDLE defined, so the notebook
runs with no copy of this repository attached and nothing to install: the rules engine is pure Python.
"""
import json
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
BUNDLE = "/tmp/ganda_grammar_bundle"
PARTS = {"src": "ganda_grammar", "scripts": "scripts"}  # repository folder → folder in the bundle


def files():
    for folder, target in PARTS.items():
        for p in sorted((REPO / folder).rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts:
                yield p, f"{BUNDLE}/{target}/{p.relative_to(REPO / folder).as_posix()}"


def cell(kind, text, n):
    c = {"cell_type": kind, "id": f"bundle{n:02d}", "metadata": {}, "source": text.splitlines(keepends=True)}
    if kind == "code":
        c |= {"execution_count": None, "outputs": []}
    return c


def version():
    try:
        rev = subprocess.run(["git", "-C", str(REPO), "describe", "--always", "--dirty"], capture_output=True, text=True,
                             check=True).stdout.strip()
        return f"commit {rev}"
    except (OSError, subprocess.CalledProcessError):
        return "no git revision"


def main(source, out=None):
    source = Path(source)
    out = Path(out) if out else source.with_name(source.stem + "_selfcontained.ipynb")
    nb = json.loads(source.read_text(encoding="utf-8"))
    bundled = list(files())
    folders = sorted({str(Path(target).parent) for _, target in bundled})
    head = [
        cell("markdown", f"""## ganda-grammar, carried by this notebook

This copy of `{source.name}` carries ganda-grammar ({version()}). The cells below write its package (`src/`, as
`ganda_grammar`) and `scripts/` to `{BUNDLE}` and put that first on `sys.path`, so the notebook needs no copy of the
repository as an input. Made by `scripts/bundle_notebook.py`; rebuild it after changing the code.
""", 0),
        cell("code", f"""import sys
from pathlib import Path

BUNDLE = Path("{BUNDLE}")
for folder in {folders!r}:
    Path(folder).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BUNDLE))
""", 1),
    ]
    for p, target in bundled:
        head.append(cell("code", f"%%writefile {target}\n{p.read_text(encoding='utf-8')}", len(head)))
    nb["cells"] = head + nb["cells"]
    out.write_text(json.dumps(nb, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"{out}: {len(bundled)} files bundled, {len(nb['cells'])} cells")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/scripts/clean_dictionary.py
"""Clean the Luganda dictionary: drop duplicates, merge near-duplicates, repair broken entries. Writes a new copy.

    python scripts/clean_dictionary.py data/luganda_dictionary.jsonl
    → data/luganda_dictionary_clean.jsonl, data/luganda_dictionary_clean.tsv, and
      data/luganda_dictionary_clean.changes.tsv (one row per change, to review)

No entry is dropped except by merging it into the entry it duplicates; its sources, meanings and spellings are kept.
In order:

1. Exact duplicates: headwords equal after Unicode normalisation, one apostrophe, single spaces and lowercase.
2. Glued rows of the English-Luganda list: its PDF columns were cut so that a headword sometimes carries the next
   rows (`decide okusalawo decorate okutona` = decide → okusalawo, decorate → okutona). Split at the English words.
3. Meanings: drop FSI cross-references (`see under .wanga`) and Luganda that spilled into the English
   (`dinner eky'eggulo` → dinner), move FSI perfect forms (`(.liisizza) feed`) to `perfect`, close brackets, and
   drop repeated meanings.
4. Split words: `oku kyusa` → okukyusa, and any phrase whose spaces-removed form is itself an entry, except
   possessives (`a maanyi`, powerful, is not amaanyi, strength).
5. Listed misspellings (src/rules/spellings.tsv): merged into their spelling, kept in `variants`.
6. Bare nouns and infinitives (kitabo, kutendeka) that only repeat the full form (ekitabo, okutendeka): merged into it,
   kept in `bare` with its own count in `bare_count`. Not when the bare form is more frequent or means something else (kye, mu, musiru).
"""
import collections
import json
import re
import sys
import unicodedata
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
try:
    from ganda_grammar.rules.base import SHAPE
except ImportError:  # not installed: load src/ as ganda_grammar
    import importlib.util
    spec = importlib.util.spec_from_file_location("ganda_grammar", REPO / "src" / "__init__.py",
                                                  submodule_search_locations=[str(REPO / "src")])
    module = importlib.util.module_from_spec(spec)
    sys.modules["ganda_grammar"] = module
    spec.loader.exec_module(module)
    from ganda_grammar.rules.base import SHAPE

FIELDS = ["headword", "pos", "noun_class", "plural", "english", "sources", "variants", "count", "attested", "bare",
          "bare_count", "perfect"]
LISTS = {"english", "sources", "variants"}
APOSTROPHE = re.compile("[’ʼ‘`]")
ELIDED = re.compile(r"^([a-z]{1,3})'([a-z]+)$")  # eky'omugaso, n'abantu
NOUN_PREFIX = re.compile(r"^(oku|okw|omu|omw|aba|eki|eky|ebi|eby|obu|obw|olu|olw|ama|emi|emy|aka|enn|emm)")
CROSS_REF = re.compile(r"^see under \.")
PERFECT = re.compile(r"^\((\.[^)]*)\)\s*")  # FSI: (.liisizza) feed


def norm(s):
    s = APOSTROPHE.sub("'", unicodedata.normalize("NFC", s))
    return " ".join(s.split()).lower()


class Cleaner:
    def __init__(self, rows):
        self.log = []
        self.D = {}
        for r in rows:
            r = {f: r.get(f, [] if f in LISTS else None) for f in FIELDS}
            hw = norm(r["headword"])
            if hw in self.D:
                self.merge(r, self.D[hw], "duplicate")
            else:
                if hw != r["headword"]:
                    self.change("normalised", r["headword"], hw)
                r["headword"] = hw
                self.D[hw] = r
        # every word used in a meaning, to tell English from Luganda
        self.english_words = collections.Counter(t for r in self.D.values() for e in r["english"]
                                                 for t in re.findall(r"[a-z]+", e.lower()))
        self.luganda_words = {h for h, r in self.D.items() if " " not in h and r["attested"]}

    def change(self, kind, before, after, note=""):
        self.log.append((kind, before, after, note))

    def merge(self, src, dst, kind, keep_as=None, before=None):
        """Fold src into dst. keep_as: the field ("variants" or "bare") that remembers src's headword."""
        for f in ("sources", "variants"):
            dst[f] += [x for x in src[f] if x not in dst[f]]
        self.add_meanings(dst, src["english"])
        for f in ("pos", "noun_class", "plural", "count", "bare", "bare_count", "perfect"):
            if dst[f] is None:
                dst[f] = src[f]
        if keep_as == "variants" and src["headword"] not in dst["variants"]:
            dst["variants"].append(src["headword"])
        elif keep_as == "bare":
            dst["bare"], dst["bare_count"] = src["headword"], src["count"]
        self.change(kind, before or src["headword"], dst["headword"])

    @staticmethod
    def add_meanings(r, meanings):
        seen = {re.sub(r"\W+", " ", e.lower()).strip() for e in r["english"]}
        for e in meanings:
            k = re.sub(r"\W+", " ", e.lower()).strip()
            if k and k not in seen:
                seen.add(k)
                r["english"].append(e)

    def is_english(self, t):
        return self.english_words[t] >= 1 and t not in self.luganda_words and "'" not in t

    def is_luganda(self, t):
        t = t.strip("(),.;:!?").lower()
        m = ELIDED.match(t)
        if m:
            return m.group(1) not in ("o", "i") and SHAPE.fullmatch(m.group(2)) is not None  # not o'clock, i'm
        return (SHAPE.fullmatch(t) is not None and self.english_words[t] == 0
                and (t in self.luganda_words or NOUN_PREFIX.match(t) is not None))

    def put(self, r, kind):
        """Add a new or rebuilt entry, merging it into an existing one with its headword."""
        if r["headword"] in self.D:
            self.merge(r, self.D[r["headword"]], kind)
        else:
            self.D[r["headword"]] = r

    # 2
    def split_glued(self):
        for hw in [h for h, r in self.D.items() if r["sources"] == ["english-luganda-list"] and " " in h]:
            toks = hw.split()
            if not any(self.is_english(t) for t in toks):
                continue
            r = self.D.pop(hw)
            runs = [[False, []]]  # [english?, tokens]
            for t in toks:
                e = self.is_english(t)
                if runs[-1][1] and runs[-1][0] != e:
                    runs.append([e, []])
                runs[-1][0] = e
                runs[-1][1].append(t)
            meanings = list(r["english"])
            pairs = []  # (luganda, [english])
            for i, (eng, words) in enumerate(runs):
                if eng and i == 0:  # the end of the row's own meaning: excuse + "me owange" = excuse me
                    tail = " ".join(words)
                    meanings = [m for m in meanings if not all(self.is_luganda(w) for w in m.split())]
                    meanings = meanings[:-1] + [f"{meanings[-1]} {tail}"] if meanings else [tail]
                elif not eng:
                    pairs.append((" ".join(words), meanings if not pairs else [" ".join(runs[i - 1][1])]))
            # a trailing English run starts the row after this one; its Luganda is lost
            parts = []
            for j, (lg, en) in enumerate(pairs):
                new = {f: ([] if f in LISTS else None) for f in FIELDS}
                new.update(headword=lg, sources=["english-luganda-list"], english=list(en))
                if j == 0:
                    new.update(pos=r["pos"], noun_class=r["noun_class"], plural=r["plural"], variants=r["variants"])
                self.put(new, "split-glued (merged)")
                parts.append(f"{lg} = {'; '.join(en)}")
            self.change("split-glued", hw, " | ".join(parts))

    # 3
    def clean_meanings(self):
        for r in self.D.values():
            out = []
            for e in r["english"]:
                before = e
                e = e.strip().lstrip("#").strip()
                m = PERFECT.match(e)
                if m:
                    r["perfect"] = r["perfect"] or m.group(1).rstrip("·").strip()
                    e = e[m.end():].strip()
                if CROSS_REF.match(e):
                    e = ""
                toks = e.split()
                k = len(toks)
                while k > 0 and self.is_luganda(toks[k - 1]):
                    k -= 1
                e = " ".join(toks[:k])
                if e.count("(") > e.count(")"):
                    e += ")"
                if e != before:
                    self.change("meaning", r["headword"], e, before)
                if e:
                    out.append(e)
            r["english"] = []
            self.add_meanings(r, out)

    # 4
    def join_split_words(self):
        for hw in [h for h in self.D if " " in h]:
            if hw not in self.D:
                continue
            joined = hw.replace(" ", "")
            if hw.startswith("a "):  # a possessive, not a split word: a maanyi (powerful) is not amaanyi
                continue
            if hw.startswith("oku ") or joined in self.D:
                r = self.D.pop(hw)
                r["headword"] = joined
                if joined in self.D:
                    self.merge(r, self.D[joined], "split-word", before=hw)
                else:
                    self.D[joined] = r
                    self.change("split-word", hw, joined)

    # 5
    def merge_misspellings(self, spellings):
        for bad, good in spellings.items():
            if bad in self.D and good in self.D:
                self.merge(self.D.pop(bad), self.D[good], "misspelling", keep_as="variants")

    # 6
    def merge_bare_forms(self):
        for hw in [h for h in self.D if " " not in h and len(h) >= 4 and h[0] not in "aeiou"]:
            for aug in "aeo":
                full = self.D.get(aug + hw)
                if full is None or hw not in self.D:
                    continue
                bare = self.D[hw]
                if bare["pos"] not in (None, full["pos"]) or (bare["count"] or 0) > (full["count"] or 0):
                    continue
                b = {e.lower() for e in bare["english"]}
                if b and not b & {e.lower() for e in full["english"]}:
                    continue
                self.merge(self.D.pop(hw), full, "bare-form", keep_as="bare")
                break

    def rows(self):
        for r in sorted(self.D.values(), key=lambda r: r["headword"]):
            r["attested"] = (r["count"] or 0) >= 2 or len(r["sources"]) >= 2
            yield r


def main(path="data/luganda_dictionary.jsonl", out=None):
    path = Path(path)
    out = Path(out) if out else path.with_name("luganda_dictionary_clean.jsonl")
    rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    spellings = dict(line.rstrip("\n").split("\t")[:2]
                     for line in (REPO / "src" / "rules" / "spellings.tsv").open(encoding="utf-8")
                     if line.strip() and not line.startswith("#"))

    c = Cleaner(rows)
    c.split_glued()
    c.clean_meanings()
    c.join_split_words()
    c.merge_misspellings(spellings)
    c.merge_bare_forms()
    clean = list(c.rows())

    with out.open("w", encoding="utf-8") as f:
        for r in clean:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with out.with_suffix(".tsv").open("w", encoding="utf-8") as f:
        f.write("\t".join(FIELDS) + "\n")
        for r in clean:
            f.write("\t".join("; ".join(x.replace(";", ",") for x in r[k]) if k in LISTS
                              else "" if r[k] is None else str(r[k]) for k in FIELDS) + "\n")
    log = out.with_suffix(".changes.tsv")
    with log.open("w", encoding="utf-8") as f:
        f.write("change\tbefore\tafter\tnote\n")
        for row in c.log:
            f.write("\t".join(str(x).replace("\t", " ") for x in row) + "\n")

    kinds = collections.Counter(k for k, *_ in c.log)
    print(f"{len(rows):,} entries → {len(clean):,} ({sum(r['attested'] for r in clean):,} attested)")
    for k, n in kinds.most_common():
        print(f"  {k:22} {n:,}")
    print(f"wrote {out}, {out.with_suffix('.tsv')}, {log}")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile /tmp/ganda_grammar_bundle/scripts/compile_luganda_text.py
"""Compile the Luganda text under a folder into chunked .txt files, the local input of notebooks/build_dictionary.ipynb.

    python scripts/compile_luganda_text.py ../ ../luganda-dictionary-inputs

Reads every .txt, .jsonl and .html file under the folder (except SKIP) and keeps the sentences that read as Luganda (the
ganda_grammar sentence check). Files are taken cleanest first (largest share of Luganda sentences); a file that is mostly
a copy of one already taken (DUPLICATE of its 5-word sequences seen before, like a re-OCR of the same book) is skipped,
and every sentence is written once. texts/luganda_text_NNN.txt get one paragraph a line, CHUNK_BYTES at most each;
texts/manifest.tsv says what each file gave, or which files were skipped as copies. sources/ gets the text layers of the
PDFs the notebook reads (pdftotext -layout), so it needs neither the PDFs nor poppler.
"""
import hashlib
import json
import re
import shutil
import subprocess
import sys
from html.parser import HTMLParser
from pathlib import Path

from ganda_grammar.rules.base import luganda_tokens, sentences

CHUNK_BYTES = 10_000_000
DUPLICATE = 0.5  # share of a file's 5-word sequences already written that makes it a copy
SKIP = {  # not Luganda (machine-translation and tokenizer experiments), or derived from the files kept here
    "minimind", "cross-lingual-group-tokenizer", "fastest-efficient-tokenizer", "brain.en", "grouptok", "grouptok-paper",
    "ganda-grammar", "luganda-dictionary-inputs", ".git", ".venv", "venv", "node_modules", "site-packages", "__pycache__",
    ".cache", ".ipynb_checkpoints",
}
RESOURCES = "gandalm/resources"
PDF_TEXT = {  # file in sources/: (PDF under RESOURCES, pdftotext page options)
    "fsi_glossary_layout.txt": ("FSI - Luganda Basic Course - Instructor and Student Text.pdf", ["-f", "364", "-l", "385"]),
    "english_luganda_layout.txt": ("English-Luganda_version_2_0.pdf", []),
}
FSI_TEXT = "FSI - Luganda Basic Course - Instructor and Student Text.txt"  # its lesson glossaries


class VisibleText(HTMLParser):  # the text of a page, without scripts and styles
    def __init__(self):
        super().__init__()
        self.parts, self.hidden = [], 0

    def handle_starttag(self, tag, attrs):
        self.hidden += tag in ("script", "style")

    def handle_endtag(self, tag):
        if tag in ("script", "style") and self.hidden:
            self.hidden -= 1

    def handle_data(self, data):
        if not self.hidden:
            self.parts.append(data)


def strings(x):  # every string in a JSON row: text fields, chat turns
    if isinstance(x, str):
        yield x
    elif isinstance(x, dict):
        for v in x.values():
            yield from strings(v)
    elif isinstance(x, list):
        for v in x:
            yield from strings(v)


def paragraphs(path):
    if path.suffix == ".jsonl":
        for line in path.open(encoding="utf-8", errors="replace"):
            try:
                yield from strings(json.loads(line))
            except ValueError:
                yield line
    elif path.suffix == ".html":
        page = VisibleText()
        page.feed(path.read_text(encoding="utf-8", errors="replace"))
        yield from "".join(page.parts).splitlines()
    else:
        yield from path.open(encoding="utf-8", errors="replace")


def files(root):
    for p in sorted(root.rglob("*")):
        if p.suffix in (".txt", ".jsonl", ".html") and p.is_file() and not SKIP.intersection(p.relative_to(root).parts):
            yield p


def luganda_paragraphs(path):
    """The file's paragraphs as lists of their Luganda sentences, and the share of its sentences that are Luganda."""
    out, total = [], 0
    for para in paragraphs(path):
        keep = []
        for s, e in sentences(para):
            total += 1
            if luganda_tokens(para, s, e):
                keep.append(re.sub(r"\s+", " ", para[s:e]).strip())
        if keep:
            out.append(keep)
    return out, sum(map(len, out)) / total if total else 0


def shingles(paras):
    words = " ".join(" ".join(p) for p in paras).lower().split()
    return {hash(tuple(words[i:i + 5])) for i in range(len(words) - 4)}


def main(root, out):
    root, out = Path(root).resolve(), Path(out)
    texts = out / "texts"
    texts.mkdir(parents=True, exist_ok=True)
    found = [(path, *luganda_paragraphs(path)) for path in files(root)]
    seen, written, manifest, n, size = set(), set(), [], 0, CHUNK_BYTES
    chunk = None
    for path, paras, share in sorted(found, key=lambda f: -f[2]):  # cleanest copy first
        if not paras:
            continue
        own = shingles(paras)
        if own and len(own & written) >= DUPLICATE * len(own):
            manifest.append((str(path.relative_to(root)), "copy", "skipped"))
            continue
        written |= own
        kept = words = 0
        for para in paras:
            keep = []
            for sent in para:
                key = hashlib.blake2b(sent.lower().encode(), digest_size=8).digest()
                if key not in seen:
                    seen.add(key)
                    keep.append(sent)
            if not keep:
                continue
            line = " ".join(keep) + "\n"
            if size + len(line.encode()) > CHUNK_BYTES:
                if chunk:
                    chunk.close()
                n += 1
                chunk, size = (texts / f"luganda_text_{n:03d}.txt").open("w", encoding="utf-8"), 0
            chunk.write(line)
            size += len(line.encode())
            kept += len(keep)
            words += len(line.split())
        if kept:
            manifest.append((str(path.relative_to(root)), kept, words))
    if chunk:
        chunk.close()
    with (texts / "manifest.tsv").open("w", encoding="utf-8") as f:
        f.write("file\tsentences\twords\n")
        f.writelines(f"{p}\t{k}\t{w}\n" for p, k, w in manifest)

    sources = out / "sources"
    sources.mkdir(exist_ok=True)
    for name, (pdf, pages) in PDF_TEXT.items():
        subprocess.run(["pdftotext", "-layout", *pages, str(root / RESOURCES / pdf), str(sources / name)], check=True)
    shutil.copy(root / RESOURCES / FSI_TEXT, sources / "fsi_course.txt")

    taken = [m for m in manifest if m[1] != "copy"]
    print(f"{len(taken)} files, {sum(k for _, k, _ in taken):,} sentences, {sum(w for *_, w in taken):,} words → {n} chunks "
          f"in {texts}; {len(manifest) - len(taken)} files skipped as copies")


if __name__ == "__main__":
    main(*sys.argv[1:])


# Luganda word list from 48 Hugging Face datasets

Reads the Luganda text columns of every dataset below and counts each word form, writing `luganda_words.tsv`:

| column | meaning |
|---|---|
| `word` | the word, lowercased |
| `count` | occurrences in human-written text |
| `datasets` | how many human-written datasets have it |
| `synthetic_count` | occurrences in machine-translated or generated text, counted apart |
| `capitalised` | share of its mid-sentence occurrences written with a capital: near 1 for names |

Only sentences that read as Luganda are counted (ganda-grammar's own test), and within a dataset each sentence counts
once. Only the listed text columns are downloaded, never the audio or images stored beside them.

On Kaggle: Settings → Internet on. A dataset that fails is reported and skipped; each finished dataset is saved
to `parts/`, so a rerun resumes where the last one stopped.


In [ ]:
import collections, json, os, pickle, re, time, traceback, urllib.parse, urllib.request
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

OUT = Path("/kaggle/working" if Path("/kaggle").exists() else ".") / "luganda_words"
PARTS = OUT / "parts"
PARTS.mkdir(parents=True, exist_ok=True)
MIN_COUNT = 2   # a word written fewer times than this (human + synthetic) is left out of the list
WORKERS = os.cpu_count() or 2

# (repo, config, Luganda text columns, synthetic). A column path: "a", "a.b" (a field), "a[]" (a list of strings, joined),
# "a[].b" (field b of each item of list a).
DATASETS = [
    # human-written
    ("henryhale/luganda-pretrain2", "default", ["text"], False),
    ("goldfish-models/fish-food", "lug_latn", ["text"], False),
    ("sagepond/gec", "default", ["tgt"], False),  # tgt only: src holds deliberately misspelled text
    ("sagepond/luganda", "default", ["luganda"], False),
    ("sagepond/luganda", "paragraphs", ["luganda"], False),
    ("michsethowusu/ganda-sentiments-corpus", "default", ["Ganda"], False),
    ("cis-lmu/Glot500", "lug_Latn", ["text"], False),
    ("michsethowusu/english-ganda_sentence-pairs_mt560", "default", ["lug"], False),
    ("google/WaxalNLP", "lug_asr", ["transcription"], False),
    ("henryhale/luganda-pretrain", "default", ["text"], False),
    ("pkyoyetera/luganda_english_dataset", "default", ["Luganda"], False),
    ("henryhale/english-luganda-translation", "default", ["luganda"], False),
    ("sulaimank/luganda_text_wiki_bukedde_radio_flores", "default", ["text"], False),
    ("aceuganda/lacuna-luganda-health-corpus", "default", ["approved_translation"], False),
    ("HuggingFaceFW/fineweb-2", "lug_Latn", ["text"], False),
    ("HuggingFaceFW/finetranslations", "lug_Latn", ["og_full_text"], False),
    ("cis-lmu/Taxi1500-RawData", "lug_Latn", ["text"], False),
    ("multilingual-tts/open-bible", "Luganda", ["text"], False),
    ("cyanic-selkie/wikianc", "lg", ["paragraph_text"], False),
    ("HPLT/HPLT2.0_cleaned", "lug_Latn", ["text"], False),
    ("AfriSpeech/madlad-91-african", "lg_clean", ["local"], False),
    ("kukuzaai/lug-cleaned-200M-tokens", "default", ["headline", "full_body"], False),
    ("KMayanja/merged_txt_eng_lug", "default", ["Luganda"], False),
    ("keystats/luganda_asr_dataset", "default", ["transcription"], False),
    ("akoksal/muri-it-language-split", "lug", ["output"], False),
    ("sbmaruf/forai_ml_masakhane_mafand", "en-lug", ["target"], False),
    ("masakhane/masakhaner2", "lug", ["tokens[]"], False),
    ("wikimedia/wikidata-title-desc", "lg", ["description"], False),
    ("allandclive/UgandaLex", "default", ["Ganda"], False),
    ("allandclive/UgandaLex2", "default", ["Ganda"], False),
    ("gimmy256/ugandan_languages", "default", ["Luganda Translation"], False),
    ("alexandrainst/multi-wiki-qa", "lg", ["context"], False),
    # machine-translated or generated
    ("henryhale/tinystories-ganda", "default", ["tgt"], True),
    ("bigscience/xP3all", "lg", ["targets"], True),
    ("michsethowusu/Code-170k-luganda", "default", ["conversations[].value"], True),
    ("yigagilbert/synthetic-parallel-external", "default", ["text_lug"], True),
    ("yigagilbert/synthetic-parallel-sunbird", "default", ["text_lug"], True),
    ("yigagilbert/synthetic-parallel-salt", "default", ["text_lug"], True),
    ("henryhale/luganda-sft-translated", "default", ["instruction_lug", "output_lug"], True),
    ("henryhale/luganda-simplestories", "default", ["text"], True),
    ("saillab/alpaca_luganda_taco", "default", ["instruction", "input", "output"], True),
    ("saillab/alpaca-luganda-cleaned", "default", ["instruction", "input", "output"], True),
    ("henryhale/luganda-ultrafeedback-dpo", "default", ["prompt", "chosen[].content", "rejected[].content"], True),
    ("henryhale/luganda-smol-smoltalk-40k", "default", ["conversations[].content"], True),
    ("henryhale/luganda-reasoning", "default", ["question", "reasoning"], True),
    ("anyantudre/waxal-linsna", "pseudo_agree_lug", ["text"], True),  # transcripts made by a speech model
    ("Tadesse/COMET_score", "eng_lug", ["Luganda"], True),
    ("gimmy256/afri-aya-vision", "default", ["question_local", "answer_local", "caption_local"], True),
]
# henryhale/luganda-pretrain2 also holds tinystories-ganda (machine-translated), already counted as synthetic above
SKIP_ROWS = {"henryhale/luganda-pretrain2": lambda row: row.get("source") == "tinystories-ganda"}
EXTRA_COLUMNS = {"henryhale/luganda-pretrain2": ["source"]}
assert len({(r, c) for r, c, *_ in DATASETS}) == len(DATASETS), "a repo and config listed twice"
print(len(DATASETS), "datasets:", sum(not d[3] for d in DATASETS), "human-written,", sum(d[3] for d in DATASETS), "synthetic")


In [ ]:
# Reading: the Hugging Face dataset viewer lists every dataset as parquet files; only the needed columns are fetched
import pyarrow.parquet as pq
from fsspec.parquet import open_parquet_file  # fetches only the byte ranges of the columns asked for

from ganda_grammar.rules.base import looks_luganda, luganda_tokens, sentences

API = "https://datasets-server.huggingface.co"


def get_json(path, **params):
    url = f"{API}/{path}?" + urllib.parse.urlencode(params)
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return json.load(r)
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * 2 ** attempt)


def parquet_files(repo, config):
    files = get_json("parquet", dataset=repo, config=config)["parquet_files"]
    assert files, "no parquet files"
    return [f["url"] for f in files]


def rows(url, columns):
    """The rows of one parquet file with only these top-level columns (audio and images beside them aren't fetched)."""
    for attempt in range(5):
        try:
            with open_parquet_file(url, columns=columns, engine="pyarrow") as f:
                pf = pq.ParquetFile(f)
                names = set(pf.schema_arrow.names)
                missing = [c for c in columns if c not in names]
                assert not missing, f"no column {missing} in {sorted(names)[:20]}"
                batches = list(pf.iter_batches(batch_size=10_000, columns=columns))
            break
        except AssertionError:
            raise
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * 2 ** attempt)
    for b in batches:
        yield from b.to_pylist()


def texts(value, path):
    """The strings at a column path: text, a.b, a[], a[].b."""
    if not path:
        if isinstance(value, str):
            yield value
        return
    head, _, rest = path.partition(".")
    is_list = head.endswith("[]")
    if head.rstrip("[]"):
        value = value.get(head.rstrip("[]")) if isinstance(value, dict) else None
    if value is None:
        return
    if is_list and not rest:  # a list of strings is one text: the words of a sentence (tokens[])
        yield " ".join(x for x in value if isinstance(x, str)) if isinstance(value, list) else ""
    elif is_list:
        for item in value if isinstance(value, list) else []:
            yield from texts(item, rest)
    else:
        yield from texts(value, rest)


In [ ]:
# Counting one dataset (run in a worker process; the result is saved to parts/)
def part_file(repo, config):
    return PARTS / (re.sub(r"[^\w.-]", "_", f"{repo}__{config}") + ".pkl")


def count_dataset(repo, config, columns, synthetic):
    start = time.time()
    counts, capitals = collections.Counter(), collections.Counter()
    seen = set()  # each sentence counts once within the dataset
    stats = collections.Counter()
    tops = sorted({c.split(".")[0].rstrip("[]") for c in columns} | set(EXTRA_COLUMNS.get(repo, [])))
    skip = SKIP_ROWS.get(repo)
    for url in parquet_files(repo, config):
        for row in rows(url, tops):
            stats["rows"] += 1
            if skip and skip(row):
                stats["rows skipped"] += 1
                continue
            for col in columns:
                for text in texts(row, col):
                    for s, e in sentences(text):
                        stats["sentences"] += 1
                        key = hash(" ".join(text[s:e].lower().split()))
                        if key in seen:
                            continue
                        seen.add(key)
                        toks = luganda_tokens(text, s, e)
                        if not toks:
                            continue
                        stats["luganda sentences"] += 1
                        for i, t in enumerate(toks):
                            w = t.text.lower()
                            if looks_luganda(w):
                                counts[w] += 1
                                if i and t.text[0].isupper():
                                    capitals[w] += 1
    stats["words"] = sum(counts.values())
    stats["seconds"] = round(time.time() - start)
    result = dict(repo=repo, config=config, synthetic=synthetic, counts=counts, capitals=capitals, stats=dict(stats))
    with open(part_file(repo, config), "wb") as f:
        pickle.dump(result, f)
    return repo, config, stats


def run(d):
    try:
        return count_dataset(*d)
    except Exception as e:
        return d[0], d[1], {"error": f"{type(e).__name__}: {str(e)[:300]}"}


In [ ]:
# Count every dataset not done yet, several at a time
todo = [d for d in DATASETS if not part_file(d[0], d[1]).exists()]
print(f"{len(DATASETS) - len(todo)} done before, {len(todo)} to count with {WORKERS} workers")
errors = {}
with ProcessPoolExecutor(WORKERS, mp_context=multiprocessing.get_context("fork")) as pool:
    jobs = [pool.submit(run, d) for d in todo]
    for job in as_completed(jobs):
        repo, config, stats = job.result()
        if "error" in stats:
            errors[f"{repo} [{config}]"] = stats["error"]
            print(f"FAILED {repo} [{config}]: {stats['error']}")
        else:
            print(f"{repo} [{config}]: {stats.get('rows', 0):,} rows, {stats.get('luganda sentences', 0):,} Luganda "
                  f"sentences, {stats.get('words', 0):,} words, {stats['seconds']}s")
print(f"{len(errors)} failed" + (": rerun this cell to retry them" if errors else ""))


In [ ]:
# Merge the datasets into one list
count, synthetic, capitals = collections.Counter(), collections.Counter(), collections.Counter()
datasets, synthetic_datasets = collections.Counter(), collections.Counter()
report = []
for repo, config, _, syn in DATASETS:
    p = part_file(repo, config)
    if not p.exists():
        report.append(dict(repo=repo, config=config, synthetic=syn, status="failed: " + errors.get(f"{repo} [{config}]", "not counted")))
        continue
    with open(p, "rb") as f:
        r = pickle.load(f)
    (synthetic if syn else count).update(r["counts"])
    (synthetic_datasets if syn else datasets).update(r["counts"].keys())
    capitals.update(r["capitals"])
    report.append(dict(repo=repo, config=config, synthetic=syn, status="ok", **r["stats"]))

words = sorted(set(count) | set(synthetic), key=lambda w: (-count[w], -synthetic[w], w))
kept = 0
with open(OUT / "luganda_words.tsv", "w", encoding="utf-8") as f:
    f.write("word\tcount\tdatasets\tsynthetic_count\tsynthetic_datasets\tcapitalised\n")
    for w in words:
        n = count[w] + synthetic[w]
        if n < MIN_COUNT:
            continue
        kept += 1
        f.write(f"{w}\t{count[w]}\t{datasets[w]}\t{synthetic[w]}\t{synthetic_datasets[w]}\t{capitals[w] / n:.2f}\n")

cols = ["repo", "config", "synthetic", "status", "rows", "rows skipped", "sentences", "luganda sentences", "words", "seconds"]
with open(OUT / "datasets.tsv", "w", encoding="utf-8") as f:
    f.write("\t".join(cols) + "\n")
    for r in report:
        f.write("\t".join(str(r.get(c, "")) for c in cols) + "\n")

human = sum(r.get("words", 0) for r in report if not r["synthetic"])
machine = sum(r.get("words", 0) for r in report if r["synthetic"])
print(f"{sum(r['status'] == 'ok' for r in report)} of {len(report)} datasets counted")
print(f"{human:,} words of human-written text, {machine:,} synthetic")
print(f"{len(words):,} distinct words, {kept:,} seen at least {MIN_COUNT} times -> {OUT / 'luganda_words.tsv'}")


In [ ]:
# A look: the commonest words, and words seen in many datasets but rarely
for w in words[:30]:
    print(f"{w:20} {count[w]:>12,} {datasets[w]:>3} datasets")
print()
spread = [w for w in words if datasets[w] >= 10 and count[w] < 50][:30]
print("in 10+ datasets but under 50 times:", ", ".join(spread))
